<a href="https://colab.research.google.com/github/lachuoiquan203/pos-sunmi-t2/blob/main/POS_cho_Sunmi_T2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import subprocess

print("⏳ 1/2: Đang chuẩn bị môi trường biên dịch cho bản v11.0...")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["ANDROID_HOME"] = "/content/android-sdk"
os.environ["ANDROID_SDK_ROOT"] = "/content/android-sdk"
os.environ["PATH"] += ":/content/flutter/bin:/content/android-sdk/platform-tools"

if not os.path.exists("/content/pos_sunmi_t2"):
    os.system("flutter create --org com.lachuoi --project-name pos_sunmi_t2 --platforms=android /content/pos_sunmi_t2 > /dev/null")

os.chdir("/content/pos_sunmi_t2")
os.system("rm -rf test")
os.system("flutter pub add sqflite:^2.3.0 path:^1.8.3 path_provider:^2.1.1 intl:^0.18.1 http:^1.1.0 sunmi_printer_plus:^2.1.3 image_picker:^1.0.4 audioplayers:^5.2.1 file_picker:^6.1.1 > /dev/null")

os.system("sed -i 's/minSdkVersion .*/minSdkVersion 21/g' android/app/build.gradle || true")
os.system("sed -i 's/minSdk .*/minSdkVersion 21/g' android/app/build.gradle || true")
os.system("sed -i 's/compileSdkVersion .*/compileSdkVersion 34/g' android/app/build.gradle || true")
os.system("sed -i 's/compileSdk .*/compileSdkVersion 34/g' android/app/build.gradle || true")
os.system('find /root/.pub-cache -name "build.gradle" -path "*sunmi_printer_plus*" -exec sed -i \'/android {/a \\    namespace "br.com.brasizza.sunmi_printer_plus"\' {} +')

LT = chr(60)
GT = chr(62)

dart_code = f"""import 'dart:async';
import 'dart:convert';
import 'dart:io';
import 'dart:math' as math;
import 'dart:typed_data';
import 'dart:ui' as ui;
import 'package:flutter/material.dart';
import 'package:flutter/services.dart';
import 'package:intl/intl.dart';
import 'package:path/path.dart' as p;
import 'package:path_provider/path_provider.dart';
import 'package:sqflite/sqflite.dart';
import 'package:http/http.dart' as http;
import 'package:image_picker/image_picker.dart';
import 'package:file_picker/file_picker.dart';
import 'package:audioplayers/audioplayers.dart';
import 'package:sunmi_printer_plus/sunmi_printer_plus.dart';
import 'package:sunmi_printer_plus/enums.dart' hide List;

const String APP_VERSION = 'v11.0';
const String APP_BUILD_INFO = 'Phiên bản 11.0 Revolution (Phân quyền PIN + Quản lý Thực đơn Thông minh + Sync 2 chiều Google Sheets)';

const Color kOchaGreen = Color(0xFF4EA63B);
const Color kOchaActionGreen = Color(0xFF418934);
const Color kOchaOccupiedRed = Color(0xFFD95338);
const Color kOchaTakeawayBlue = Color(0xFF5C9CE6);
const Color kOchaBottomBar = Color(0xFF2E353E);
const Color kOchaBottomActive = Color(0xFF3E4752);
const Color kOchaBgGrey = Color(0xFFF2F3F5);

// Hàm bỏ dấu tiếng Việt để tìm kiếm món ăn siêu tốc (VD: gõ "db ko cha" -> khớp "ĐB ko chả")
String removeDiacritics(String str) {{
  var result = str.toLowerCase();
  const viet = 'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ';
  const eng  = 'aaaaaaaaaaaaaaaaaeeeeeeeeeeeiiiiiooooooooooooooooouuuuuuuuuuuyyyyyd';
  for (int i = 0; i {LT} viet.length; i++) {{
    result = result.replaceAll(viet[i], eng[i]);
  }}
  return result;
}}

void main() async {{
  WidgetsFlutterBinding.ensureInitialized();
  await SystemChrome.setPreferredOrientations([
    DeviceOrientation.landscapeLeft,
    DeviceOrientation.landscapeRight,
  ]);
  await SystemChrome.setEnabledSystemUIMode(SystemUiMode.immersiveSticky);
  runApp(const OchaRevolutionV11App());
}}

class OchaRevolutionV11App extends StatelessWidget {{
  const OchaRevolutionV11App({{super.key}});

  @override
  Widget build(BuildContext context) {{
    return MaterialApp(
      title: 'Lá Chuối Quán - Ocha POS $APP_VERSION',
      debugShowCheckedModeBanner: false,
      theme: ThemeData(
        primaryColor: kOchaGreen,
        scaffoldBackgroundColor: kOchaBgGrey,
        useMaterial3: false,
      ),
      home: const OchaPosMainScreen(),
    );
  }}
}}

// ==================== 1. ÂM THANH KÉT TIỀN (HỖ TRỢ TẮT / MẶC ĐỊNH / FILE TỰ TẢI) ====================
class CashDrawerBellSound {{
  static final AudioPlayer _player = AudioPlayer();
  static String? _defaultWavPath;
  static String soundMode = 'DEFAULT';
  static String customSoundPath = '';
  static double customStartOffsetSec = 0.0;

  static Future{LT}void{GT} playSound({{String? forceMode}}) async {{
    final String mode = forceMode ?? soundMode;
    if (mode == 'OFF') return;
    try {{
      await _player.stop();
      if (mode == 'CUSTOM' && customSoundPath.isNotEmpty && File(customSoundPath).existsSync()) {{
        await _player.play(
          DeviceFileSource(customSoundPath),
          volume: 1.0,
          position: Duration(milliseconds: (customStartOffsetSec * 1000).round()),
        );
        return;
      }}
      if (_defaultWavPath == null || !File(_defaultWavPath!).existsSync()) {{
        _defaultWavPath = await _generateExactOchaRecordedSoundWav();
      }}
      await _player.play(DeviceFileSource(_defaultWavPath!), volume: 1.0);
    }} catch (_) {{
      SystemSound.play(SystemSoundType.alert);
    }}
  }}

  static Future{LT}String{GT} _generateExactOchaRecordedSoundWav() async {{
    final dir = await getTemporaryDirectory();
    final file = File('${{dir.path}}/ocha_real_ting_v11.wav');
    const int sampleRate = 44100;
    const double duration = 0.92;
    final int numSamples = (sampleRate * duration).toInt();
    final ByteData data = ByteData(44 + numSamples * 2);

    void writeStr(int offset, String s) {{
      for (int i = 0; i {LT} s.length; i++) data.setUint8(offset + i, s.codeUnitAt(i));
    }}

    writeStr(0, 'RIFF');
    data.setUint32(4, 36 + numSamples * 2, Endian.little);
    writeStr(8, 'WAVE');
    writeStr(12, 'fmt ');
    data.setUint32(16, 16, Endian.little);
    data.setUint16(20, 1, Endian.little);
    data.setUint16(22, 1, Endian.little);
    data.setUint32(24, sampleRate, Endian.little);
    data.setUint32(28, sampleRate * 2, Endian.little);
    data.setUint16(32, 2, Endian.little);
    data.setUint16(34, 16, Endian.little);
    writeStr(36, 'data');
    data.setUint32(40, numSamples * 2, Endian.little);

    final math.Random rng = math.Random(42);
    for (int i = 0; i {LT} numSamples; i++) {{
      final double t = i / sampleRate;
      double wave = 0.0;
      if (t {LT} 0.045) {{
        final double envL = math.exp(-85.0 * t);
        wave += envL * (0.45 * math.sin(2 * math.pi * 240.0 * t) + (rng.nextDouble() * 0.6 - 0.3));
      }}
      if (t {GT}= 0.035) {{
        final double t1 = t - 0.035;
        wave += (1.0 - math.exp(-900.0 * t1)) * math.exp(-18.0 * t1) *
            (0.42 * math.sin(2 * math.pi * 2180.0 * t1) + 0.25 * math.sin(2 * math.pi * 3450.0 * t1));
      }}
      if (t {GT}= 0.088) {{
        final double t2 = t - 0.088;
        final double att2 = 1.0 - math.exp(-1200.0 * t2);
        wave += att2 * (
          0.54 * math.exp(-5.2 * t2) * math.sin(2 * math.pi * 2840.0 * t2) +
          0.32 * math.exp(-9.5 * t2) * math.sin(2 * math.pi * 4185.0 * t2) +
          0.20 * math.exp(-18.0 * t2) * math.sin(2 * math.pi * 5890.0 * t2)
        );
      }}
      int sample = (wave * 30000).round().clamp(-32768, 32767);
      data.setInt16(44 + i * 2, sample, Endian.little);
    }}
    await file.writeAsBytes(data.buffer.asUint8List(), flush: true);
    return file.path;
  }}
}}

// ==================== 2. HÌNH VẼ LÁ CỜ RUY-BĂNG & BÀN 4 GHẾ OCHA ====================
class OchaRibbonClipper extends CustomClipper{LT}Path{GT} {{
  @override
  Path getClip(Size size) {{
    final Path path = Path();
    path.lineTo(0, size.height);
    path.lineTo(size.width / 2, size.height - 5.5);
    path.lineTo(size.width, size.height);
    path.lineTo(size.width, 0);
    path.close();
    return path;
  }}
  @override
  bool shouldReclip(covariant CustomClipper{LT}Path{GT} oldClipper) =""" + f"""> false;
}}

class OchaTableShapePainter extends CustomPainter {{
  final Color color;
  final bool hasOrder;
  final bool showBillBadge;

  OchaTableShapePainter({{required this.color, required this.hasOrder, this.showBillBadge = false}});

  @override
  void paint(Canvas canvas, Size size) {{
    final double w = size.width;
    final double h = size.height;
    final double side = math.min(w, h) * 0.64;
    final Rect tableRect = Rect.fromCenter(center: Offset(w / 2, h / 2), width: side, height: side);

    canvas.drawRect(tableRect, Paint()..color = Colors.white..style = PaintingStyle.fill);
    canvas.drawRect(tableRect, Paint()..color = color..style = PaintingStyle.stroke..strokeWidth = 1.8);

    if (showBillBadge) {{
      final Rect bRect = Rect.fromCenter(center: Offset(tableRect.right - 2, tableRect.top + 2), width: 11, height: 13);
      canvas.drawRRect(RRect.fromRectAndRadius(bRect, const Radius.circular(1.5)), Paint()..color = color);
      final Paint lp = Paint()..color = Colors.white..strokeWidth = 1.2;
      canvas.drawLine(Offset(bRect.left + 2.5, bRect.top + 4), Offset(bRect.right - 2.5, bRect.top + 4), lp);
      canvas.drawLine(Offset(bRect.left + 2.5, bRect.top + 7.5), Offset(bRect.right - 2.5, bRect.top + 7.5), lp);
    }}

    final Paint cp = Paint()..color = color..style = PaintingStyle.fill;
    const double cw = 13.0, ch = 9.0, gap = 3.2;
    canvas.drawArc(Rect.fromCenter(center: Offset(w / 2, tableRect.top - gap), width: cw, height: ch), math.pi, math.pi, true, cp);
    canvas.drawArc(Rect.fromCenter(center: Offset(w / 2, tableRect.bottom + gap), width: cw, height: ch), 0, math.pi, true, cp);
    canvas.drawArc(Rect.fromCenter(center: Offset(tableRect.left - gap, h / 2), width: ch, height: cw), math.pi / 2, math.pi, true, cp);
    canvas.drawArc(Rect.fromCenter(center: Offset(tableRect.right + gap, h / 2), width: ch, height: cw), -math.pi / 2, math.pi, true, cp);
  }}

  @override
  bool shouldRepaint(covariant OchaTableShapePainter oldDelegate) =""" + f""">
      oldDelegate.color != color || oldDelegate.hasOrder != hasOrder || oldDelegate.showBillBadge != showBillBadge;
}}

// ==================== 3. CƠ SỞ DỮ LIỆU SQLITE (NÂNG CẤP USERS/RBAC & DYNAMIC CATEGORIES) ====================
class LocalDb {{
  static Database? _db;

  static Future{LT}Database{GT} get db async {{
    if (_db != null) return _db!;
    _db = await _init();
    return _db!;
  }}

  static Future{LT}Database{GT} _init() async {{
    final dbPath = await getDatabasesPath();
    final path = p.join(dbPath, 'lachuoi_ocha_v8.db'); // Giữ nguyên file DB để không mất đơn/món cũ

    final database = await openDatabase(path, version: 1, onCreate: (db, version) async {{}});
    await _ensureAllTablesAndMigrations(database);
    return database;
  }}

  static Future{LT}void{GT} _ensureAllTablesAndMigrations(Database db) async {{
    await db.execute('''
      CREATE TABLE IF NOT EXISTS categories (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE,
        icon_code TEXT,
        sort_order INTEGER DEFAULT 0
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        pin TEXT,
        role TEXT,
        can_view_report INTEGER DEFAULT 1,
        can_view_activity INTEGER DEFAULT 1,
        can_edit_menu INTEGER DEFAULT 1,
        can_delete_order INTEGER DEFAULT 1
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS products (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        bottom_tab TEXT,
        short_code TEXT,
        name TEXT,
        price INTEGER,
        image_path TEXT,
        in_stock INTEGER DEFAULT 1
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS table_areas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        area_name TEXT UNIQUE
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS dining_tables (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        area TEXT,
        table_code TEXT,
        table_name TEXT
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        receipt_no TEXT,
        takeaway_code TEXT,
        order_type TEXT,
        area TEXT,
        table_name TEXT,
        guest_count INTEGER,
        staff_name TEXT,
        note TEXT,
        items_json TEXT,
        discount INTEGER,
        surcharge INTEGER,
        total INTEGER,
        payment_method TEXT,
        status TEXT,
        prebill_printed INTEGER DEFAULT 0,
        start_time TEXT,
        end_time TEXT,
        synced INTEGER DEFAULT 0
      )
    ''');

    await db.execute('''
      CREATE TABLE IF NOT EXISTS settings (
        key TEXT PRIMARY KEY,
        value TEXT
      )
    ''');

    // Khởi tạo Danh mục động nếu chưa có
    final catCnt = Sqflite.firstIntValue(await db.rawQuery('SELECT COUNT(*) FROM categories')) ?? 0;
    if (catCnt == 0) {{
      final defaultCats = [
        {{'name': 'CHUẨN', 'icon_code': 'soup', 'sort_order': 1}},
        {{'name': 'Em bé', 'icon_code': 'child', 'sort_order': 2}},
        {{'name': 'NƯỚC', 'icon_code': 'drink', 'sort_order': 3}},
        {{'name': 'THÊM', 'icon_code': 'extra', 'sort_order': 4}},
        {{'name': 'BÁNH-BÚN', 'icon_code': 'cake', 'sort_order': 5}},
      ];
      for (var c in defaultCats) await db.insert('categories', c, conflictAlgorithm: ConflictAlgorithm.ignore);
    }}

    // Khởi tạo Tài khoản Admin & Nhân viên mặc định nếu chưa có
    final usrCnt = Sqflite.firstIntValue(await db.rawQuery('SELECT COUNT(*) FROM users')) ?? 0;
    if (usrCnt == 0) {{
      await db.insert('users', {{
        'name': 'Le Xuan Dong (Admin)',
        'pin': '1234',
        'role': 'ADMIN',
        'can_view_report': 1,
        'can_view_activity': 1,
        'can_edit_menu': 1,
        'can_delete_order': 1,
      }});
      await db.insert('users', {{
        'name': 'Thu Ngân Ca Sáng',
        'pin': '1111',
        'role': 'STAFF',
        'can_view_report': 0, // Không cho xem báo cáo doanh thu
        'can_view_activity': 1,
        'can_edit_menu': 0,
        'can_delete_order': 0,
      }});
      await db.insert('users', {{
        'name': 'Nhân Viên Phục Vụ',
        'pin': '2222',
        'role': 'STAFF',
        'can_view_report': 0,
        'can_view_activity': 0,
        'can_edit_menu': 0,
        'can_delete_order': 0,
      }});
    }}

    final setCnt = Sqflite.firstIntValue(await db.rawQuery('SELECT COUNT(*) FROM settings')) ?? 0;
    if (setCnt == 0) {{
      await db.insert('settings', {{'key': 'store_name', 'value': 'HKD La Chuoi Quan'}});
      await db.insert('settings', {{'key': 'store_address', 'value': '203 Nguyễn Văn Trỗi, P.4, Vũng Tàu'}});
      await db.insert('settings', {{'key': 'store_phone', 'value': '0328000203'}});
      await db.insert('settings', {{'key': 'staff_name', 'value': 'Le Xuan Dong'}});
      await db.insert('settings', {{'key': 'webhook_url', 'value': ''}});
      await db.insert('settings', {{'key': 'sound_mode', 'value': 'DEFAULT'}});
      await db.insert('settings', {{'key': 'custom_sound_path', 'value': ''}});
      await db.insert('settings', {{'key': 'custom_sound_offset', 'value': '0.0'}});
      await db.insert('settings', {{'key': 'require_pin_on_start', 'value': '0'}});
    }}

    final areaCnt = Sqflite.firstIntValue(await db.rawQuery('SELECT COUNT(*) FROM table_areas')) ?? 0;
    if (areaCnt == 0) {{
      for (var a in ['Bàn', 'Bếp', 'Hông']) {{
        await db.insert('table_areas', {{'area_name': a}}, conflictAlgorithm: ConflictAlgorithm.ignore);
      }}
      for (int i = 1; i {LT}= 30; i++) {{
        await db.insert('dining_tables', {{'area': 'Bàn', 'table_code': '$i', 'table_name': 'Bàn - $i'}});
      }}
      for (int i = 1; i {LT}= 4; i++) {{
        await db.insert('dining_tables', {{'area': 'Bếp', 'table_code': 'B$i', 'table_name': 'Bếp - $i'}});
      }}
      for (int i = 1; i {LT}= 6; i++) {{
        await db.insert('dining_tables', {{'area': 'Hông', 'table_code': 'H$i', 'table_name': 'Hông - $i'}});
      }}
    }}

    final prodCnt = Sqflite.firstIntValue(await db.rawQuery('SELECT COUNT(*) FROM products')) ?? 0;
    if (prodCnt == 0) {{
      final defaultItems = [
        {{'bottom_tab': 'CHUẨN', 'short_code': 'BT', 'name': 'Bình Thường', 'price': 45000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'BT', 'name': 'BT Không chả', 'price': 45000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'BT', 'name': 'BT ít chả', 'price': 45000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'BT', 'name': 'BT', 'price': 45000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'BT', 'name': 'BT ko rau ko chả', 'price': 45000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'ĐB', 'name': 'Đặc biệt', 'price': 65000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'ĐB', 'name': 'ĐB ko chả', 'price': 65000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'ĐB', 'name': 'ĐB 1 chả 1 nem', 'price': 65000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'ĐB', 'name': 'ĐB ko rau', 'price': 65000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'ĐB', 'name': 'ĐB ko rau ko chả', 'price': 65000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TT', 'name': 'Trứng thịt', 'price': 50000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TT', 'name': 'TT ko chả', 'price': 50000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TT', 'name': 'TT ít chả', 'price': 50000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TT', 'name': 'TT ko rau', 'price': 50000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TT', 'name': 'TT ko rau ko chả', 'price': 50000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TC', 'name': 'Thịt chả', 'price': 40000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TC', 'name': 'TC ko chả', 'price': 40000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TC', 'name': 'TC ít chả', 'price': 40000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TC', 'name': 'TC korau', 'price': 40000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TC', 'name': 'TC korau ko chả', 'price': 40000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TÔ', 'name': 'Tôm thịt', 'price': 55000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TÔ', 'name': 'Tôm thịt ko chả', 'price': 55000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TÔ', 'name': 'Tôm thịt ít chả', 'price': 55000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TÔ', 'name': 'Tôm thịt ko rau', 'price': 55000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'CHUẨN', 'short_code': 'TÔ', 'name': 'Tôm thịt korau ko chả', 'price': 55000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'Em bé', 'short_code': 'EB', 'name': 'Em bé', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'Em bé', 'short_code': 'EB', 'name': 'E bé ko chả', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'Em bé', 'short_code': 'EB', 'name': 'E bé ko hành', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'Em bé', 'short_code': 'EB', 'name': 'E bé trứng', 'price': 20000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'NƯỚC', 'short_code': 'ĐL', 'name': 'Nước đinh lăng', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'NƯỚC', 'short_code': 'SS', 'name': 'Sương sâm / Rong biển', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'NƯỚC', 'short_code': 'SĐ', 'name': 'Sữa đậu nành', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'NƯỚC', 'short_code': 'NN', 'name': 'Nước ngọt', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'THÊM', 'short_code': 'CL', 'name': 'Chả lụa thêm', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'THÊM', 'short_code': 'CN', 'name': 'Chả nướng / Nem', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'THÊM', 'short_code': 'TR', 'name': 'Trứng thêm', 'price': 10000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'THÊM', 'short_code': 'BC', 'name': 'Bánh cuốn thêm', 'price': 15000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'BÍ', 'name': 'Bánh ít lá gai', 'price': 10000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'BB', 'name': 'Bánh bò thốt nốt', 'price': 25000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'DL', 'name': 'Bánh da lợn', 'price': 20000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'KM', 'name': 'Bánh khoai mì nướng', 'price': 25000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'KC', 'name': 'Khoai mỳ chuối hấp', 'price': 20000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'CĐ', 'name': 'Chè đậu trắng', 'price': 20000, 'image_path': '', 'in_stock': 1}},
        {{'bottom_tab': 'BÁNH-BÚN', 'short_code': 'BT', 'name': 'Bánh tét', 'price': 65000, 'image_path': '', 'in_stock': 1}},
      ];
      for (var it in defaultItems) await db.insert('products', it);
    }}
  }}
}}

// ==================== 4. MÁY IN BITMAP K80 TỐC ĐỘ CAO (NON-BLOCKING) ====================
class OchaPrinter {{
  static bool isBound = false;

  static Future{LT}void{GT} init() async {{
    try {{
      bool? b = await SunmiPrinter.bindingPrinter();
      isBound = b ?? false;
    }} catch (_) {{
      isBound = false;
    }}
  }}

  static Future{LT}Uint8List{GT} _renderExactOchaReceiptBitmap({{
    required String storeName,
    required String storePhone,
    required String receiptNo,
    required String orderType,
    required String tableName,
    required String staffName,
    required String startTimeStr,
    required String endTimeStr,
    required List items,
    required int discount,
    required int surcharge,
    required int total,
    required String paymentMethod,
    required String note,
    required bool isKitchen,
  }}) async {{
    const double width = 576.0;
    final fmt = NumberFormat("#,###", "en_US");
    final ui.PictureRecorder recorder = ui.PictureRecorder();
    final Canvas canvas = Canvas(recorder);

    canvas.drawRect(const Rect.fromLTWH(0, 0, width, 2800), Paint()..color = Colors.white);
    double y = 16.0;

    void drawCenteredText(String text, double fontSize, FontWeight weight) {{
      final tp = TextPainter(
        text: TextSpan(text: text, style: TextStyle(color: Colors.black, fontSize: fontSize, fontWeight: weight, fontFamily: 'sans-serif')),
        textDirection: ui.TextDirection.ltr,
        textAlign: TextAlign.center,
      )..layout(maxWidth: width);
      tp.paint(canvas, Offset((width - tp.width) / 2, y));
      y += tp.height + 6;
    }}

    void drawLeftRightText(String left, String right, double fontSize, {{FontWeight leftW = FontWeight.w400, FontWeight rightW = FontWeight.w400}}) {{
      final tpL = TextPainter(
        text: TextSpan(text: left, style: TextStyle(color: Colors.black, fontSize: fontSize, fontWeight: leftW)),
        textDirection: ui.TextDirection.ltr,
      )..layout(maxWidth: width * 0.62);
      final tpR = TextPainter(
        text: TextSpan(text: right, style: TextStyle(color: Colors.black, fontSize: fontSize, fontWeight: rightW)),
        textDirection: ui.TextDirection.ltr,
      )..layout(maxWidth: width * 0.45);
      tpL.paint(canvas, Offset(12, y));
      tpR.paint(canvas, Offset(width - 12 - tpR.width, y));
      y += math.max(tpL.height, tpR.height) + 8;
    }}

    void drawDottedLine() {{
      y += 6;
      final Paint dotPaint = Paint()..color = Colors.black87..strokeWidth = 1.8;
      for (double x = 14; x {LT} width - 14; x += 9) canvas.drawCircle(Offset(x, y), 1.1, dotPaint);
      y += 14;
    }}

    if (isKitchen) {{
      drawCenteredText('PHIẾU BẾP - $tableName', 32, FontWeight.w700);
      drawDottedLine();
      drawLeftRightText('Bàn / Đơn:', tableName, 24, rightW: FontWeight.w700);
      drawLeftRightText('NV gọi:', staffName, 22);
      drawLeftRightText('Giờ gọi:', endTimeStr, 22);
      if (note.isNotEmpty) drawLeftRightText('Ghi chú HĐ:', note, 22, rightW: FontWeight.w700);
      drawDottedLine();
      for (var it in items) {{
        final String name = it['name'].toString();
        final int qty = it['qty'] as int;
        final String mods = (it['modifiers'] ?? '').toString();
        final String itemNote = (it['note'] ?? '').toString();
        drawLeftRightText('$qty x  $name', '', 28, leftW: FontWeight.w700);
        if (mods.isNotEmpty) drawLeftRightText('   + $mods', '', 22);
        if (itemNote.isNotEmpty) drawLeftRightText('   (*$itemNote)', '', 22);
      }}
      drawDottedLine();
    }} else {{
      drawCenteredText(storeName, 29, FontWeight.w500);
      drawCenteredText('SĐT: $storePhone', 23, FontWeight.w400);
      y += 10;
      drawCenteredText('BIÊN LAI', 27, FontWeight.w600);
      drawDottedLine();

      drawLeftRightText('Số biên lai:', receiptNo, 23);
      drawLeftRightText('$orderType:', tableName, 23);
      drawLeftRightText('Phục vụ bởi:', staffName, 23);
      drawLeftRightText('Giờ vào:', startTimeStr, 23);
      drawLeftRightText('Giờ ra:', endTimeStr, 23);
      if (note.isNotEmpty) drawLeftRightText('Ghi chú:', note, 22);
      drawDottedLine();

      void drawFourColRow(String c1, String c2, String c3, String c4, double fSize, FontWeight fw) {{
        final tp1 = TextPainter(text: TextSpan(text: c1, style: TextStyle(color: Colors.black, fontSize: fSize, fontWeight: fw)), textDirection: ui.TextDirection.ltr)..layout(maxWidth: 250);
        final tp2 = TextPainter(text: TextSpan(text: c2, style: TextStyle(color: Colors.black, fontSize: fSize, fontWeight: fw)), textDirection: ui.TextDirection.ltr)..layout(maxWidth: 95);
        final tp3 = TextPainter(text: TextSpan(text: c3, style: TextStyle(color: Colors.black, fontSize: fSize, fontWeight: fw)), textDirection: ui.TextDirection.ltr)..layout(maxWidth: 65);
        final tp4 = TextPainter(text: TextSpan(text: c4, style: TextStyle(color: Colors.black, fontSize: fSize, fontWeight: fw)), textDirection: ui.TextDirection.ltr)..layout(maxWidth: 150);

        tp1.paint(canvas, Offset(12, y));
        tp2.paint(canvas, Offset(265, y));
        tp3.paint(canvas, Offset(385 - tp3.width / 2, y));
        tp4.paint(canvas, Offset(width - 12 - tp4.width, y));
        y += math.max(tp1.height, tp4.height) + 4;
      }}

      drawFourColRow('Mặt hàng', 'Đơn vị', 'SL', 'Số tiền', 24, FontWeight.w600);
      y += 6;

      for (var it in items) {{
        final String name = it['name'].toString();
        final int qty = it['qty'] as int;
        final int unitPrice = (it['price'] as int) + ((it['modifier_price'] ?? 0) as int);
        final int lineTotal = qty * unitPrice;
        final String mods = (it['modifiers'] ?? '').toString();
        final String itemNote = (it['note'] ?? '').toString();

        drawFourColRow(name, '', '$qty', fmt.format(lineTotal), 23, FontWeight.w400);
        final tpUnit = TextPainter(
          text: TextSpan(text: fmt.format(unitPrice), style: const TextStyle(color: Colors.black, fontSize: 23, fontWeight: FontWeight.w400)),
          textDirection: ui.TextDirection.ltr,
        )..layout();
        tpUnit.paint(canvas, Offset(12, y));
        y += tpUnit.height + 8;

        if (mods.isNotEmpty) drawLeftRightText('  + $mods', '', 20);
        if (itemNote.isNotEmpty) drawLeftRightText('  (*$itemNote)', '', 20);
      }}

      y += 6;
      drawLeftRightText('Tổng', fmt.format(total), 25, leftW: FontWeight.w600, rightW: FontWeight.w600);
      if (discount != 0) drawLeftRightText('Giảm giá', '-${{fmt.format(discount)}}', 22);
      if (surcharge != 0) drawLeftRightText('Phụ phí', '+${{fmt.format(surcharge)}}', 22);
      drawDottedLine();

      drawLeftRightText('Tổng', '${{fmt.format(total)}}₫', 28, leftW: FontWeight.w600, rightW: FontWeight.w600);
      drawLeftRightText(paymentMethod, '${{fmt.format(total)}}₫', 23);

      y += 16;
      drawCenteredText('Cảm ơn & chúc bạn một ngày tuyệt vời!', 22, FontWeight.w400);
      drawCenteredText('Powered by Ocha', 21, FontWeight.w400);
      y += 20;
    }}

    final ui.Picture picture = recorder.endRecording();
    final ui.Image img = await picture.toImage(width.toInt(), y.ceil() + 10);
    final ByteData? pngBytes = await img.toByteData(format: ui.ImageByteFormat.png);
    return pngBytes!.buffer.asUint8List();
  }}

  static Future{LT}void{GT} printBill({{
    required String storeName,
    required String storeAddress,
    required String storePhone,
    required String receiptNo,
    required String orderType,
    required String tableName,
    required int guestCount,
    required String staffName,
    required String startTime,
    required List items,
    required int discount,
    required int surcharge,
    required int total,
    required String paymentMethod,
    required String note,
    required String printMode,
    required bool openDrawer,
  }}) async {{
    if (!isBound) await init();

    String fmtDate(String raw) {{
      try {{
        return DateFormat('dd-MM-yyyy HH:mm').format(DateTime.parse(raw));
      }} catch (_) {{
        return DateFormat('dd-MM-yyyy HH:mm').format(DateTime.now());
      }}
    }}

    final String inTime = fmtDate(startTime);
    final String outTime = DateFormat('dd-MM-yyyy HH:mm').format(DateTime.now());
    final bool isKitchenMode = printMode == 'KITCHEN' || printMode == 'KITCHEN_ADD';

    try {{
      await SunmiPrinter.initPrinter();
      await SunmiPrinter.startTransactionPrint(true);

      final Uint8List bitmapBytes = await _renderExactOchaReceiptBitmap(
        storeName: storeName,
        storePhone: storePhone,
        receiptNo: receiptNo,
        orderType: orderType,
        tableName: tableName,
        staffName: staffName,
        startTimeStr: inTime,
        endTimeStr: outTime,
        items: items,
        discount: discount,
        surcharge: surcharge,
        total: total,
        paymentMethod: paymentMethod,
        note: note,
        isKitchen: isKitchenMode,
      );

      await SunmiPrinter.setAlignment(SunmiPrintAlign.CENTER);
      await SunmiPrinter.printImage(bitmapBytes);
      await SunmiPrinter.lineWrap(3);
      await SunmiPrinter.cut();

      if (printMode == 'PAID' || openDrawer) {{
        CashDrawerBellSound.playSound();
        await SunmiPrinter.openDrawer();
      }}
      await SunmiPrinter.exitTransactionPrint(true);
    }} catch (e) {{
      if (printMode == 'PAID' || openDrawer) CashDrawerBellSound.playSound();
    }}
  }}
}}

// ==================== 5. MÀN HÌNH CHÍNH (TÍCH HỢP LOGIN PIN, PHÂN QUYỀN & QUẢN LÝ THÔNG MINH) ====================
class OchaPosMainScreen extends StatefulWidget {{
  const OchaPosMainScreen({{super.key}});

  @override
  State{LT}OchaPosMainScreen{GT} createState() =""" + f"""> _OchaPosMainScreenState();
}}

class _OchaPosMainScreenState extends State{LT}OchaPosMainScreen{GT} {{
  final fmt = NumberFormat("#,###", "en_US");
  final ImagePicker _picker = ImagePicker();
  Timer? _clockTimer;

  // Trạng thái Đăng nhập & Phân quyền User (RBAC)
  bool isLockedPinScreen = false;
  String enteredPin = '';
  String pinErrorMsg = '';
  List allUsers = [];
  Map{LT}String, dynamic{GT} currentUser = {{
    'id': 1,
    'name': 'Le Xuan Dong (Admin)',
    'pin': '1234',
    'role': 'ADMIN',
    'can_view_report': 1,
    'can_view_activity': 1,
    'can_edit_menu': 1,
    'can_delete_order': 1,
  }};

  String currentBottomTab = 'Hóa đơn';
  String currentTopAreaTab = 'Hóa đơn';
  String fullScreenView = 'NONE'; // 'NONE', 'HOAT_DONG', 'BAO_CAO', 'CAI_DAT', 'SMART_MENU'
  bool isEditGridMode = false;
  bool isDeleteBillMode = false;
  bool isEditTableMode = false;

  String selectedLibrarySection = 'ROOT';
  String librarySearch = '';
  String posFastSearch = ''; // Ô tìm kiếm không dấu siêu tốc ngay trên lưới bán hàng
  String settingsSubTab = 'USERS'; // 'USERS', 'SOUND', 'STORE', 'TABLES', 'VERSION'

  String storeName = 'HKD La Chuoi Quan';
  String storeAddress = '203 Nguyễn Văn Trỗi, P.4, Vũng Tàu';
  String storePhone = '0328000203';
  String webhookUrl = '';

  String soundMode = 'DEFAULT';
  String customSoundPath = '';
  double customSoundOffset = 0.0;
  bool requirePinOnStart = false;

  List allCategories = [];
  List allProducts = [];
  List{LT}String{GT} allAreaNames = ['Bàn', 'Bếp', 'Hông'];
  List allDiningTables = [];
  List openOrders = [];
  List paidOrders = [];
  dynamic selectedActivityOrder;

  int? activeOrderId;
  String activeReceiptNo = '11002255';
  String activeTakeawayCode = '';
  String activeOrderType = 'Dùng tại quán';
  String activeArea = 'Bàn';
  String activeTableName = 'Chọn bàn';
  int activeGuestCount = 1;
  String activeOrderNote = '';
  String activeStartTime = '';
  int activePrebillPrinted = 0;
  List cart = [];
  List undoCartSnapshot = [];
  int activeDiscount = 0;
  int activeSurcharge = 0;

  final List{LT}Map{LT}String, dynamic{GT}{GT} ochaModifiers = [
    {{'name': 'Ít chả', 'price': 0}},
    {{'name': 'Không chả', 'price': 0}},
    {{'name': 'Không rau', 'price': 0}},
    {{'name': 'Không hành', 'price': 0}},
    {{'name': 'Nước mắm riêng', 'price': 0}},
    {{'name': 'Làm nóng kỹ', 'price': 0}},
    {{'name': 'Thêm chả lụa', 'price': 15000}},
    {{'name': 'Thêm nem/chả nướng', 'price': 15000}},
    {{'name': 'Thêm trứng', 'price': 10000}},
    {{'name': 'Thêm bánh cuốn', 'price': 15000}},
  ];

  @override
  void initState() {{
    super.initState();
    OchaPrinter.init();
    _loadData(isFirstBoot: true);
    _clockTimer = Timer.periodic(const Duration(seconds: 20), (_) {{
      if (mounted && currentBottomTab == 'Hóa đơn') setState(() {{}});
    }});
  }}

  @override
  void dispose() {{
    _clockTimer?.cancel();
    super.dispose();
  }}

  bool get isAdmin =""" + f"""> currentUser['role'] == 'ADMIN';
  bool get canViewReport =""" + f"""> isAdmin || (currentUser['can_view_report'] ?? 0) == 1;
  bool get canViewActivity =""" + f"""> isAdmin || (currentUser['can_view_activity'] ?? 0) == 1;
  bool get canEditMenu =""" + f"""> isAdmin || (currentUser['can_edit_menu'] ?? 0) == 1;
  bool get canDeleteOrder =""" + f"""> isAdmin || (currentUser['can_delete_order'] ?? 0) == 1;
  String get currentStaffName =""" + f"""> currentUser['name'].toString().replaceAll(' (Admin)', '');

  Future{LT}void{GT} _loadData({{bool isFirstBoot = false}}) async {{
    final db = await LocalDb.db;
    final cats = await db.query('categories', orderBy: 'sort_order ASC, id ASC');
    final usrs = await db.query('users', orderBy: 'id ASC');
    final prods = await db.query('products', orderBy: 'id ASC');
    final areasRows = await db.query('table_areas', orderBy: 'id ASC');
    final tbls = await db.query('dining_tables', orderBy: 'id ASC');
    final opens = await db.query('orders', where: "status = 'OPEN'", orderBy: 'id ASC');
    final paids = await db.query('orders', where: "status = 'PAID'", orderBy: 'id DESC');
    final sets = await db.query('settings');

    for (var s in sets) {{
      if (s['key'] == 'store_name') storeName = s['value'].toString();
      if (s['key'] == 'store_address') storeAddress = s['value'].toString();
      if (s['key'] == 'store_phone') storePhone = s['value'].toString();
      if (s['key'] == 'webhook_url') webhookUrl = s['value'].toString();
      if (s['key'] == 'sound_mode') soundMode = s['value'].toString();
      if (s['key'] == 'custom_sound_path') customSoundPath = s['value'].toString();
      if (s['key'] == 'custom_sound_offset') {{
        customSoundOffset = double.tryParse(s['value'].toString()) ?? 0.0;
      }}
      if (s['key'] == 'require_pin_on_start') {{
        requirePinOnStart = s['value'].toString() == '1';
      }}
    }}

    CashDrawerBellSound.soundMode = soundMode;
    CashDrawerBellSound.customSoundPath = customSoundPath;
    CashDrawerBellSound.customStartOffsetSec = customSoundOffset;

    final loadedAreas = areasRows.map((r) =""" + f"""> r['area_name'].toString()).toList();
    if (loadedAreas.isEmpty) loadedAreas.add('Bàn');

    setState(() {{
      allCategories = cats;
      allUsers = usrs;
      if (usrs.isNotEmpty && isFirstBoot) {{
        currentUser = Map{LT}String, dynamic{GT}.from(usrs.first);
        if (requirePinOnStart) isLockedPinScreen = true;
      }} else if (usrs.isNotEmpty) {{
        final updatedMe = usrs.where((u) =""" + f"""> u['id'] == currentUser['id']).toList();
        if (updatedMe.isNotEmpty) currentUser = Map{LT}String, dynamic{GT}.from(updatedMe.first);
      }}
      allProducts = prods;
      allAreaNames = loadedAreas;
      allDiningTables = tbls;
      openOrders = opens;
      paidOrders = paids;
      if (selectedActivityOrder == null && paids.isNotEmpty) {{
        selectedActivityOrder = paids.first;
      }}
    }});
  }}

  Future{LT}void{GT} _saveSetting(String k, String v) async {{
    final db = await LocalDb.db;
    await db.insert('settings', {{'key': k, 'value': v}}, conflictAlgorithm: ConflictAlgorithm.replace);
  }}

  // Kiểm tra quyền truy cập: Nếu nhân viên bị khóa quyền xem Báo cáo, bật hộp thoại hỏi mã PIN Admin!
  Future{LT}bool{GT} _verifyPermissionOrAskAdminPin(bool hasPerm, String featureName) async {{
    if (hasPerm) return true;
    String pinInput = '';
    final bool? ok = await showDialog{LT}bool{GT}(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: Text('🔒 Yêu cầu quyền truy cập: $featureName'),
        content: Column(
          mainAxisSize: MainAxisSize.min,
          children: [
            Text('Tài khoản [${{currentUser['name']}}] không được cấp quyền "$featureName".\nVui lòng nhập Mã PIN của Chủ quán (Admin) để mở:'),
            const SizedBox(height: 12),
            TextField(
              autofocus: true,
              obscureText: true,
              keyboardType: TextInputType.number,
              maxLength: 4,
              decoration: const InputDecoration(
                labelText: 'Mã PIN Admin (Mặc định: 1234)',
                border: OutlineInputBorder(),
              ),
              onChanged: (v) =""" + f"""> pinInput = v.trim(),
            ),
          ],
        ),
        actions: [
          TextButton(onPressed: () =""" + f"""> Navigator.pop(context, false), child: const Text('Hủy')),
          ElevatedButton(
            style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen),
            onPressed: () {{
              final adminMatch = allUsers.any((u) =""" + f"""> u['role'] == 'ADMIN' && u['pin'].toString() == pinInput);
              Navigator.pop(context, adminMatch);
            }},
            child: const Text('Xác nhận'),
          ),
        ],
      ),
    );
    if (ok != true && mounted) {{
      ScaffoldMessenger.of(context).showSnackBar(
        const SnackBar(backgroundColor: Colors.red, content: Text('Mã PIN Admin không đúng hoặc bạn không có quyền!')),
      );
    }}
    return ok == true;
  }}

  // ==================== ĐỒNG BỘ 2 CHIỀU DANH SÁCH MÓN HÀNG VỚI GOOGLE SHEETS & CSV ====================
  Future{LT}void{GT} _exportMenuToGoogleSheets() async {{
    if (webhookUrl.isEmpty) {{
      ScaffoldMessenger.of(context).showSnackBar(
        const SnackBar(backgroundColor: Colors.red, content: Text('Vui lòng nhập Link Google Apps Script Webhook trong Cài đặt trước!')),
      );
      return;
    }}
    try {{
      final res = await http.post(
        Uri.parse(webhookUrl),
        headers: {{'Content-Type': 'application/json'}},
        body: jsonEncode({{
          'action': 'EXPORT_MENU',
          'categories': allCategories,
          'products': allProducts,
        }}),
      ).timeout(const Duration(seconds: 12));

      if (res.statusCode == 200 || res.statusCode == 302) {{
        if (mounted) {{
          ScaffoldMessenger.of(context).showSnackBar(
            const SnackBar(backgroundColor: kOchaGreen, content: Text('✅ Đã xuất toàn bộ Danh mục & Món ăn lên Google Sheets thành công!')),
          );
        }}
      }}
    }} catch (e) {{
      if (mounted) {{
        ScaffoldMessenger.of(context).showSnackBar(
          SnackBar(backgroundColor: Colors.red, content: Text('Lỗi kết nối Google Sheets: $e')),
        );
      }}
    }}
  }}

  Future{LT}void{GT} _importMenuFromGoogleSheets() async {{
    if (webhookUrl.isEmpty) {{
      ScaffoldMessenger.of(context).showSnackBar(
        const SnackBar(backgroundColor: Colors.red, content: Text('Vui lòng nhập Link Google Apps Script Webhook trong Cài đặt trước!')),
      );
      return;
    }}
    try {{
      final uri = Uri.parse(webhookUrl.contains('?') ? '$webhookUrl&action=GET_MENU' : '$webhookUrl?action=GET_MENU');
      final res = await http.get(uri).timeout(const Duration(seconds: 12));
      if (res.statusCode == 200) {{
        final decoded = jsonDecode(res.body);
        final List items = decoded is List ? decoded : (decoded['products'] as List? ?? []);
        if (items.isEmpty) {{
          if (mounted) {{
            ScaffoldMessenger.of(context).showSnackBar(
              const SnackBar(content: Text('Sheet Thực đơn trống hoặc chưa có dữ liệu!')),
            );
          }}
          return;
        }}
        final db = await LocalDb.db;
        await db.delete('products');
        final Set{LT}String{GT} seenCats = {{}};
        int sortIdx = 1;
        for (var it in items) {{
          final String tab = (it['bottom_tab'] ?? it['category'] ?? 'CHUẨN').toString().trim();
          final String code = (it['short_code'] ?? 'BT').toString().trim();
          final String name = (it['name'] ?? '').toString().trim();
          final int price = int.tryParse(it['price'].toString().replaceAll(',', '').replaceAll('.', '')) ?? 0;
          final int stock = int.tryParse((it['in_stock'] ?? '1').toString()) ?? 1;
          if (name.isEmpty) continue;
          if (!seenCats.contains(tab)) {{
            seenCats.add(tab);
            await db.insert('categories', {{'name': tab, 'icon_code': 'food', 'sort_order': sortIdx++}}, conflictAlgorithm: ConflictAlgorithm.ignore);
          }}
          await db.insert('products', {{
            'bottom_tab': tab,
            'short_code': code,
            'name': name,
            'price': price,
            'image_path': (it['image_path'] ?? '').toString(),
            'in_stock': stock,
          }});
        }}
        await _loadData();
        if (mounted) {{
          ScaffoldMessenger.of(context).showSnackBar(
            SnackBar(backgroundColor: kOchaGreen, content: Text('✅ Đã Import thành công ${{items.length}} món ăn từ Google Sheets!')),
          );
        }}
      }}
    }} catch (e) {{
      if (mounted) {{
        ScaffoldMessenger.of(context).showSnackBar(
          SnackBar(backgroundColor: Colors.red, content: Text('Lỗi Import từ Google Sheets: $e')),
        );
      }}
    }}
  }}

  // Xuất & Nhập Thực đơn qua file CSV nội bộ / USB
  Future{LT}void{GT} _exportMenuToCsvFile() async {{
    final dir = await getExternalStorageDirectory();
    final f = File('${{dir!.path}}/ThucDon_LaChuoi.csv');
    final sb = StringBuffer('\\uFEFFDanhMuc,MaVietTat,TenMon,DonGia,ConHang\\n');
    for (var p in allProducts) {{
      sb.writeln('"${{p['bottom_tab']}}","${{p['short_code']}}","${{p['name']}}",${{p['price']}},${{p['in_stock'] ?? 1}}');
    }}
    await f.writeAsString(sb.toString());
    if (mounted) {{
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(backgroundColor: kOchaGreen, content: Text('Đã xuất file Thực đơn CSV tại: ${{f.path}}')),
      );
    }}
  }}

  Future{LT}void{GT} _importMenuFromCsvFile() async {{
    try {{
      final res = await FilePicker.platform.pickFiles(type: FileType.custom, allowedExtensions: ['csv', 'txt']);
      if (res == null || res.files.single.path == null) return;
      final content = await File(res.files.single.path!).readAsString();
      final lines = const LineSplitter().convert(content);
      if (lines.length {LT}= 1) return;

      final db = await LocalDb.db;
      int count = 0;
      for (int i = 1; i {LT} lines.length; i++) {{
        final raw = lines[i].trim();
        if (raw.isEmpty) continue;
        final parts = raw.split(',').map((s) =""" + f"""> s.replaceAll('"', '').trim()).toList();
        if (parts.length {LT} 4) continue;
        final String cat = parts[0].isEmpty ? 'CHUẨN' : parts[0];
        final String code = parts[1].isEmpty ? 'BT' : parts[1];
        final String name = parts[2];
        final int price = int.tryParse(parts[3]) ?? 0;
        final int stock = parts.length {GT} 4 ? (int.tryParse(parts[4]) ?? 1) : 1;

        await db.insert('categories', {{'name': cat, 'icon_code': 'food', 'sort_order': 10}}, conflictAlgorithm: ConflictAlgorithm.ignore);
        await db.insert('products', {{
          'bottom_tab': cat,
          'short_code': code,
          'name': name,
          'price': price,
          'image_path': '',
          'in_stock': stock,
        }});
        count++;
      }}
      await _loadData();
      if (mounted) {{
        ScaffoldMessenger.of(context).showSnackBar(
          SnackBar(backgroundColor: kOchaGreen, content: Text('✅ Đã thêm $count món từ file CSV!')),
        );
      }}
    }} catch (e) {{
      debugPrint('CSV import error: $e');
    }}
  }}

  Future{LT}String?{GT} _pickAndSaveCustomAudioFile() async {{
    try {{
      final FilePickerResult? result = await FilePicker.platform.pickFiles(
        type: FileType.custom,
        allowedExtensions: ['mp3', 'm4a', 'wav', 'aac', 'ogg'],
      );
      if (result == null || result.files.single.path == null) return null;
      final String sourcePath = result.files.single.path!;
      final dir = await getApplicationDocumentsDirectory();
      final folder = Directory('${{dir.path}}/ocha_sounds');
      if (!await folder.exists()) await folder.create(recursive: true);
      final String ext = p.extension(sourcePath).isEmpty ? '.m4a' : p.extension(sourcePath);
      final String targetPath = '${{folder.path}}/custom_ting_${{DateTime.now().millisecondsSinceEpoch}}$ext';
      await File(sourcePath).copy(targetPath);
      return targetPath;
    }} catch (_) {{
      return null;
    }}
  }}

  Map{LT}String, String{GT} _generateNextTakeawayInfo() {{
    int idx = 1;
    while (true) {{
      final name = 'Số ${{idx.toString().padLeft(3, '0')}}';
      final code = 'T${{(11000 + idx)}}';
      bool exists = openOrders.any((o) =""" + f"""> o['table_name'] == name && o['id'] != activeOrderId);
      if (!exists) return {{'name': name, 'code': code}};
      idx++;
    }}
  }}

  int get cartSubtotal {{
    int s = 0;
    for (var it in cart) {{
      int unit = (it['price'] as int) + ((it['modifier_price'] ?? 0) as int);
      s += unit * (it['qty'] as int);
    }}
    return s;
  }}

  int get cartTotal {{
    int t = cartSubtotal - activeDiscount + activeSurcharge;
    return t.isNegative ? 0 : t;
  }}

  int get cartItemCount {{
    int c = 0;
    for (var it in cart) c += (it['qty'] as int);
    return c;
  }}

  void _saveUndoSnapshot() {{
    undoCartSnapshot = jsonDecode(jsonEncode(cart));
  }}

  void _startNewOrder({{String? tableName, String? area, String? orderType}}) {{
    final randNo = (11002200 + DateTime.now().millisecondsSinceEpoch % 9000).toString();
    final firstCat = allCategories.isNotEmpty ? allCategories.first['name'].toString() : 'CHUẨN';
    setState(() {{
      activeOrderId = null;
      activeReceiptNo = randNo;
      activeTakeawayCode = '';
      activeArea = area ?? (allAreaNames.isNotEmpty ? allAreaNames.first : 'Bàn');
      activeTableName = tableName ?? 'Chọn bàn';
      activeOrderType = orderType ?? 'Dùng tại quán';
      activeGuestCount = 1;
      activeOrderNote = '';
      activePrebillPrinted = 0;
      activeStartTime = DateFormat('yyyy-MM-dd HH:mm:ss').format(DateTime.now());
      cart = [];
      activeDiscount = 0;
      activeSurcharge = 0;
      currentBottomTab = firstCat;
    }});
  }}

  void _openExistingOrder(dynamic ord) {{
    final raw = jsonDecode(ord['items_json'].toString());
    final restored = [];
    for (var it in raw) {{
      restored.add({{
        'id': it['id'],
        'name': it['name'],
        'price': it['price'],
        'qty': it['qty'],
        'printed_qty': it['printed_qty'] ?? it['qty'],
        'modifiers': it['modifiers'] ?? '',
        'modifier_price': it['modifier_price'] ?? 0,
        'note': it['note'] ?? '',
      }});
    }}
    final firstCat = allCategories.isNotEmpty ? allCategories.first['name'].toString() : 'CHUẨN';
    setState(() {{
      activeOrderId = ord['id'] as int;
      activeReceiptNo = (ord['receipt_no'] ?? '11002255').toString();
      activeTakeawayCode = (ord['takeaway_code'] ?? '').toString();
      activeOrderType = (ord['order_type'] ?? 'Dùng tại quán').toString();
      activeArea = (ord['area'] ?? 'Bàn').toString();
      activeTableName = ord['table_name'].toString();
      activeGuestCount = (ord['guest_count'] ?? 1) as int;
      activeOrderNote = (ord['note'] ?? '').toString();
      activeStartTime = ord['start_time'].toString();
      activePrebillPrinted = (ord['prebill_printed'] ?? 0) as int;
      activeDiscount = (ord['discount'] ?? 0) as int;
      activeSurcharge = (ord['surcharge'] ?? 0) as int;
      cart = restored;
      currentBottomTab = firstCat;
    }});
  }}

  void _addProductToCart(dynamic prod) {{
    if ((prod['in_stock'] ?? 1) == 0) {{
      ScaffoldMessenger.of(context).showSnackBar(
        SnackBar(backgroundColor: Colors.red.shade700, content: Text('Món [${{prod['name']}}] hiện đang HẾT HÀNG!'), duration: const Duration(seconds: 1)),
      );
      return;
    }}
    _saveUndoSnapshot();
    setState(() {{
      if (activeStartTime.isEmpty) activeStartTime = DateFormat('yyyy-MM-dd HH:mm:ss').format(DateTime.now());
      int idx = cart.indexWhere((c) =""" + f"""> c['name'] == prod['name'] && (c['modifiers'] ?? '') == '' && (c['note'] ?? '') == '');
      if (idx != -1) {{
        cart[idx]['qty'] = (cart[idx]['qty'] as int) + 1;
      }} else {{
        cart.add({{
          'id': prod['id'],
          'name': prod['name'],
          'price': prod['price'],
          'qty': 1,
          'printed_qty': 0,
          'modifiers': '',
          'modifier_price': 0,
          'note': '',
        }});
      }}
    }});
  }}

  Future{LT}void{GT} _handlePressSaveButton() async {{
    if (cart.isEmpty) return;
    if (activeTableName == 'Chọn bàn') {{
      if (activeOrderType == 'Mang về') {{
        final tk = _generateNextTakeawayInfo();
        setState(() {{
          activeTableName = tk['name']!;
          activeTakeawayCode = tk['code']!;
          activeArea = 'Mang về';
        }});
      }} else {{
        final chosen = await _showSelectTableVisualModal(forSavingNow: true);
        if (chosen == null || chosen.isEmpty) return;
      }}
    }}
    await _saveOpenOrderAndPrintKitchen();
  }}

  // TỐC ĐỘ 0 GIÂY CHỜ: Lưu DB và trả màn hình về Hóa đơn ngay lập tức, việc in K80 chạy ngầm (unawaited)!
  Future{LT}void{GT} _saveOpenOrderAndPrintKitchen({{bool markPrebill = false}}) async {{
    if (cart.isEmpty) return;
    final db = await LocalDb.db;
    final now = DateFormat('yyyy-MM-dd HH:mm:ss').format(DateTime.now());

    String finalTableName = activeTableName;
    if (finalTableName == 'Chọn bàn' || finalTableName == 'Mang về') {{
      final tk = _generateNextTakeawayInfo();
      finalTableName = tk['name']!;
      activeTakeawayCode = tk['code']!;
      activeOrderType = 'Mang về';
      activeArea = 'Mang về';
    }}

    final List itemsToPrintKitchen = [];
    for (var it in cart) {{
      final int currentQty = it['qty'] as int;
      final int alreadyPrinted = (it['printed_qty'] ?? 0) as int;
      final int diffQty = currentQty - alreadyPrinted;
      if (diffQty {GT} 0) {{
        final copyItem = Map{LT}String, dynamic{GT}.from(it);
        copyItem['qty'] = diffQty;
        itemsToPrintKitchen.add(copyItem);
      }}
      it['printed_qty'] = currentQty;
    }}

    final List snapshotCart = jsonDecode(jsonEncode(cart));
    final String snapReceiptNo = activeReceiptNo;
    final String snapOrderType = activeOrderType;
    final int snapGuest = activeGuestCount;
    final String snapStart = activeStartTime.isEmpty ? now : activeStartTime;
    final int snapDisc = activeDiscount;
    final int snapSur = activeSurcharge;
    final int snapTotal = cartTotal;
    final String snapNote = activeOrderNote;
    final bool isAddingToExistingOrder = activeOrderId != null;

    final row = {{
      'receipt_no': snapReceiptNo,
      'takeaway_code': activeTakeawayCode,
      'order_type': snapOrderType,
      'area': activeArea,
      'table_name': finalTableName,
      'guest_count': snapGuest,
      'staff_name': currentStaffName,
      'note': snapNote,
      'items_json': jsonEncode(snapshotCart),
      'discount': snapDisc,
      'surcharge': snapSur,
      'total': snapTotal,
      'payment_method': 'Tiền mặt',
      'status': 'OPEN',
      'prebill_printed': markPrebill ? 1 : activePrebillPrinted,
      'start_time': snapStart,
      'end_time': '',
      'synced': 0,
    }};

    if (activeOrderId == null) {{
      await db.insert('orders', row);
    }} else {{
      await db.update('orders', row, where: 'id = ?', whereArgs: [activeOrderId]);
    }}

    // Giải phóng giao diện ngay lập tức (0ms delay)
    await _loadData();
    setState(() {{
      activeOrderId = null;
      cart = [];
      activeTableName = 'Chọn bàn';
      activeOrderType = 'Dùng tại quán';
      currentTopAreaTab = 'Hóa đơn';
      currentBottomTab = 'Hóa đơn';
    }});

    // Đẩy lệnh in bếp chạy ngầm hoàn toàn
    final List kitchenList = itemsToPrintKitchen.isNotEmpty ? itemsToPrintKitchen : snapshotCart;
    unawaited(OchaPrinter.printBill(
      storeName: storeName,
      storeAddress: storeAddress,
      storePhone: storePhone,
      receiptNo: snapReceiptNo,
      orderType: snapOrderType,
      tableName: finalTableName,
      guestCount: snapGuest,
      staffName: currentStaffName,
      startTime: snapStart,
      items: kitchenList,
      discount: snapDisc,
      surcharge: snapSur,
      total: snapTotal,
      paymentMethod: 'Tiền mặt',
      note: snapNote,
      printMode: isAddingToExistingOrder && itemsToPrintKitchen.isNotEmpty ? 'KITCHEN_ADD' : 'KITCHEN',
      openDrawer: false,
    ));
  }}

  Future{LT}void{GT} _payOrderNow({{dynamic directOpenOrder, String method = 'Tiền mặt'}}) async {{
    final db = await LocalDb.db;
    final now = DateFormat('yyyy-MM-dd HH:mm:ss').format(DateTime.now());

    if (directOpenOrder != null) {{
      final rawItems = jsonDecode(directOpenOrder['items_json'].toString());
      await db.update(
        'orders',
        {{'status': 'PAID', 'end_time': now, 'payment_method': method, 'staff_name': currentStaffName}},
        where: 'id = ?',
        whereArgs: [directOpenOrder['id']],
      );
      await _loadData();

      unawaited(OchaPrinter.printBill(
        storeName: storeName,
        storeAddress: storeAddress,
        storePhone: storePhone,
        receiptNo: (directOpenOrder['receipt_no'] ?? '').toString(),
        orderType: (directOpenOrder['order_type'] ?? 'Dùng tại quán').toString(),
        tableName: directOpenOrder['table_name'].toString(),
        guestCount: (directOpenOrder['guest_count'] ?? 1) as int,
        staffName: currentStaffName,
        startTime: directOpenOrder['start_time'].toString(),
        items: rawItems,
        discount: (directOpenOrder['discount'] ?? 0) as int,
        surcharge: (directOpenOrder['surcharge'] ?? 0) as int,
        total: directOpenOrder['total'] as int,
        paymentMethod: method,
        note: (directOpenOrder['note'] ?? '').toString(),
        printMode: 'PAID',
        openDrawer: true,
      ));

      if (webhookUrl.isNotEmpty) {{
        unawaited(_syncToGoogleSheet(directOpenOrder['id'] as int, directOpenOrder, rawItems));
      }}
      return;
    }}

    if (cart.isEmpty) return;
    String tbl = activeTableName;
    if (tbl == 'Chọn bàn') {{
      final tk = _generateNextTakeawayInfo();
      tbl = tk['name']!;
      activeTakeawayCode = tk['code']!;
    }}

    final List snapCart = jsonDecode(jsonEncode(cart));
    final String snapReceipt = activeReceiptNo;
    final String snapType = activeOrderType;
    final int snapGuest = activeGuestCount;
    final String snapStart = activeStartTime.isEmpty ? now : activeStartTime;
    final int snapDisc = activeDiscount;
    final int snapSur = activeSurcharge;
    final int snapTot = cartTotal;
    final String snapNote = activeOrderNote;

    final row = {{
      'receipt_no': snapReceipt,
      'takeaway_code': activeTakeawayCode,
      'order_type': snapType,
      'area': activeArea,
      'table_name': tbl,
      'guest_count': snapGuest,
      'staff_name': currentStaffName,
      'note': snapNote,
      'items_json': jsonEncode(snapCart),
      'discount': snapDisc,
      'surcharge': snapSur,
      'total': snapTot,
      'payment_method': method,
      'status': 'PAID',
      'prebill_printed': 1,
      'start_time': snapStart,
      'end_time': now,
      'synced': 0,
    }};

    int id;
    if (activeOrderId == null) {{
      id = await db.insert('orders', row);
    }} else {{
      id = activeOrderId!;
      await db.update('orders', row, where: 'id = ?', whereArgs: [id]);
    }}

    await _loadData();
    setState(() {{
      activeOrderId = null;
      cart = [];
      activeTableName = 'Chọn bàn';
      activeDiscount = 0;
      activeSurcharge = 0;
      currentBottomTab = 'Hóa đơn';
    }});

    unawaited(OchaPrinter.printBill(
      storeName: storeName,
      storeAddress: storeAddress,
      storePhone: storePhone,
      receiptNo: snapReceipt,
      orderType: snapType,
      tableName: tbl,
      guestCount: snapGuest,
      staffName: currentStaffName,
      startTime: snapStart,
      items: snapCart,
      discount: snapDisc,
      surcharge: snapSur,
      total: snapTot,
      paymentMethod: method,
      note: snapNote,
      printMode: 'PAID',
      openDrawer: true,
    ));

    if (webhookUrl.isNotEmpty) {{
      unawaited(_syncToGoogleSheet(id, row, snapCart));
    }}
  }}

  Future{LT}void{GT} _syncToGoogleSheet(int id, dynamic orderRow, List items) async {{
    try {{
      final res = await http.post(
        Uri.parse(webhookUrl),
        headers: {{'Content-Type': 'application/json'}},
        body: jsonEncode({{
          'order_id': id,
          'created_at': orderRow['end_time'] ?? orderRow['start_time'],
          'table_name': orderRow['table_name'],
          'payment_method': orderRow['payment_method'],
          'total': orderRow['total'],
          'staff_name': currentStaffName,
          'items': items,
        }}),
      ).timeout(const Duration(seconds: 8));
      if (res.statusCode == 200 || res.statusCode == 302) {{
        final db = await LocalDb.db;
        await db.update('orders', {{'synced': 1}}, where: 'id = ?', whereArgs: [id]);
      }}
    }} catch (_) {{}}
  }}

  Future{LT}String?{GT} _pickAndSaveDishImage() async {{
    try {{
      final XFile? picked = await _picker.pickImage(source: ImageSource.gallery, maxWidth: 500, maxHeight: 500, imageQuality: 85);
      if (picked == null) return null;
      final dir = await getApplicationDocumentsDirectory();
      final folder = Directory('${{dir.path}}/ocha_images');
      if (!await folder.exists()) await folder.create(recursive: true);
      final ext = p.extension(picked.path).isEmpty ? '.jpg' : p.extension(picked.path);
      final target = '${{folder.path}}/img_${{DateTime.now().millisecondsSinceEpoch}}$ext';
      await File(picked.path).copy(target);
      return target;
    }} catch (_) {{
      return null;
    }}
  }}

  // ==================== GIAO DIỆN MÀN HÌNH KHÓA PIN 4 SỐ CHUẨN OCHA ====================
  Widget _buildOchaPinLockScreen() {{
    void onDigitTap(String d) {{
      setState(() {{
        pinErrorMsg = '';
        if (enteredPin.length {LT} 4) enteredPin += d;
        if (enteredPin.length == 4) {{
          final match = allUsers.where((u) =""" + f"""> u['pin'].toString() == enteredPin).toList();
          if (match.isNotEmpty) {{
            currentUser = Map{LT}String, dynamic{GT}.from(match.first);
            enteredPin = '';
            isLockedPinScreen = false;
          }} else {{
            pinErrorMsg = 'Mã PIN không đúng! (Gợi ý: Admin 1234, Thu ngân 1111)';
            enteredPin = '';
          }}
        }}
      }});
    }}

    return Scaffold(
      backgroundColor: kOchaBottomBar,
      body: Center(
        child: Container(
          width: 420,
          padding: const EdgeInsets.all(28),
          decoration: BoxDecoration(
            color: Colors.white,
            borderRadius: BorderRadius.circular(12),
            boxShadow: const [BoxShadow(color: Colors.black26, blurRadius: 16)],
          ),
          child: Column(
            mainAxisSize: MainAxisSize.min,
            children: [
              Container(
                padding: const EdgeInsets.symmetric(horizontal: 14, vertical: 6),
                decoration: BoxDecoration(color: kOchaGreen, borderRadius: BorderRadius.circular(6)),
                child: Text('$storeName • OCHA $APP_VERSION', style: const TextStyle(color: Colors.white, fontWeight: FontWeight.bold, fontSize: 16)),
              ),
              const SizedBox(height: 14),
              const Text('Nhập mã PIN nhân viên (4 số)', style: TextStyle(fontSize: 17, fontWeight: FontWeight.w600)),
              const SizedBox(height: 16),
              // 4 chấm tròn biểu thị mã PIN
              Row(
                mainAxisAlignment: MainAxisAlignment.center,
                children: List.generate(4, (i) {{
                  final bool filled = i {LT} enteredPin.length;
                  return Container(
                    margin: const EdgeInsets.symmetric(horizontal: 10),
                    width: 18,
                    height: 18,
                    decoration: BoxDecoration(
                      shape: BoxShape.circle,
                      color: filled ? kOchaGreen : Colors.transparent,
                      border: Border.all(color: kOchaGreen, width: 2),
                    ),
                  );
                }}),
              ),
              if (pinErrorMsg.isNotEmpty) ...[
                const SizedBox(height: 10),
                Text(pinErrorMsg, style: const TextStyle(color: Colors.red, fontSize: 13, fontWeight: FontWeight.bold), textAlign: TextAlign.center),
              ],
              const SizedBox(height: 20),
              // Bàn phím số 3x4
              GridView.count(
                shrinkWrap: true,
                crossAxisCount: 3,
                childAspectRatio: 1.6,
                mainAxisSpacing: 10,
                crossAxisSpacing: 10,
                children: [
                  ...['1', '2', '3', '4', '5', '6', '7', '8', '9'].map((d) =""" + f"""> OutlinedButton(
                        style: OutlinedButton.styleFrom(side: BorderSide(color: Colors.grey.shade300)),
                        onPressed: () =""" + f"""> onDigitTap(d),
                        child: Text(d, style: const TextStyle(fontSize: 22, fontWeight: FontWeight.bold, color: Colors.black87)),
                      )),
                  OutlinedButton(
                    onPressed: () =""" + f"""> setState(() =""" + f"""> enteredPin = ''),
                    child: const Text('Xóa', style: TextStyle(color: Colors.red, fontWeight: FontWeight.bold)),
                  ),
                  OutlinedButton(
                    onPressed: () =""" + f"""> onDigitTap('0'),
                    child: const Text('0', style: TextStyle(fontSize: 22, fontWeight: FontWeight.bold, color: Colors.black87)),
                  ),
                  OutlinedButton(
                    onPressed: () {{
                      if (enteredPin.isNotEmpty) {{
                        setState(() =""" + f"""> enteredPin = enteredPin.substring(0, enteredPin.length - 1));
                      }}
                    }},
                    child: const Icon(Icons.backspace_outlined, color: Colors.black54),
                  ),
                ],
              ),
              const SizedBox(height: 12),
              const Text('Mặc định: Chủ quán (1234) • Thu ngân (1111) • Phục vụ (2222)', style: TextStyle(color: Colors.grey, fontSize: 12)),
            ],
          ),
        ),
      ),
    );
  }}

  IconData _getCategoryIcon(String code) {{
    switch (code) {{
      case 'soup': return Icons.soup_kitchen_outlined;
      case 'child': return Icons.child_care;
      case 'drink': return Icons.wine_bar;
      case 'extra': return Icons.bakery_dining;
      case 'cake': return Icons.icecream_outlined;
      default: return Icons.restaurant_menu;
    }}
  }}

  @override
  Widget build(BuildContext context) {{
    if (isLockedPinScreen) return _buildOchaPinLockScreen();
    if (fullScreenView == 'HOAT_DONG') return _buildActivityScreen();
    if (fullScreenView == 'CAI_DAT') return _buildSettingsScreen();
    if (fullScreenView == 'BAO_CAO') return _buildReportScreen();
    if (fullScreenView == 'SMART_MENU') return _buildSmartMenuSpreadsheetScreen();

    return Scaffold(
      backgroundColor: kOchaBgGrey,
      body: Column(
        children: [
          Expanded(
            child: currentBottomTab == 'Hóa đơn'
                ? _buildOpenBillsAndTablesView()
                : Row(
                    children: [
                      Expanded(
                        flex: 64,
                        child: currentBottomTab == 'Thư viện'
                            ? _buildLibraryLeftPane()
                            : _buildProductGrid5x5(),
                      ),
                      const VerticalDivider(width: 1, thickness: 1, color: Color(0xFFD0D5DD)),
                      Expanded(
                        flex: 36,
                        child: _buildRightOrderPanel(),
                      ),
                    ],
                  ),
          ),

          // THANH ĐIỀU HƯỚNG DƯỚI ĐÁY ĐỘNG 100% + HIỂN THỊ TÀI KHOẢN ĐANG CA TRỰC
          Container(
            height: 64,
            color: kOchaBottomBar,
            child: Row(
              children: [
                InkWell(
                  onTap: _showNineDotsPopupMenu,
                  child: Container(
                    width: 82,
                    alignment: Alignment.center,
                    child: Column(
                      mainAxisAlignment: MainAxisAlignment.center,
                      children: const [
                        Icon(Icons.more_horiz, color: Colors.white70, size: 26),
                        Text(APP_VERSION, style: TextStyle(color: Color(0xFF76FF03), fontSize: 10, fontWeight: FontWeight.bold)),
                      ],
                    ),
                  ),
                ),
                _buildBottomTabItem('Hóa đơn', Icons.attach_money, badge: openOrders.length),
                // Các tab Danh mục món hàng động từ bảng categories
                ...allCategories.map((c) {{
                  final String cName = c['name'].toString();
                  final IconData cIcon = _getCategoryIcon((c['icon_code'] ?? '').toString());
                  return _buildBottomTabItem(cName, cIcon);
                }}),
                _buildBottomTabItem('Thư viện', Icons.library_books_outlined),
                // Nút hiển thị User hiện tại & Đổi ca / Khóa PIN nhanh ở góc dưới phải
                InkWell(
                  onTap: () =""" + f"""> setState(() {{
                    enteredPin = '';
                    pinErrorMsg = '';
                    isLockedPinScreen = true;
                  }}),
                  child: Container(
                    padding: const EdgeInsets.symmetric(horizontal: 12),
                    color: Colors.black26,
                    child: Column(
                      mainAxisAlignment: MainAxisAlignment.center,
                      children: [
                        Icon(isAdmin ? Icons.verified_user : Icons.person, color: isAdmin ? const Color(0xFF76FF03) : Colors.amberAccent, size: 18),
                        const SizedBox(height: 2),
                        Text(
                          currentStaffName,
                          maxLines: 1,
                          overflow: TextOverflow.ellipsis,
                          style: const TextStyle(color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold),
                        ),
                        const Text('Đổi ca / Khóa', style: TextStyle(color: Colors.white54, fontSize: 9)),
                      ],
                    ),
                  ),
                ),
              ],
            ),
          ),
        ],
      ),
    );
  }}

  Widget _buildBottomTabItem(String tabName, IconData icon, {{int badge = 0}}) {{
    final bool isSelected = currentBottomTab == tabName;
    return Expanded(
      child: InkWell(
        onTap: () =""" + f"""> setState(() =""" + f"""> currentBottomTab = tabName),
        child: Container(
          color: isSelected ? kOchaBottomActive : Colors.transparent,
          child: Stack(
            alignment: Alignment.center,
            children: [
              Column(
                mainAxisAlignment: MainAxisAlignment.center,
                children: [
                  Icon(icon, color: isSelected ? Colors.white : Colors.white60, size: 22),
                  const SizedBox(height: 4),
                  Text(
                    tabName,
                    maxLines: 1,
                    overflow: TextOverflow.ellipsis,
                    style: TextStyle(
                      color: isSelected ? Colors.white : Colors.white60,
                      fontSize: 12,
                      fontWeight: isSelected ? FontWeight.w600 : FontWeight.normal,
                    ),
                  ),
                ],
              ),
              if (badge != 0)
                Positioned(
                  top: 6,
                  right: 14,
                  child: Container(
                    padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 1),
                    decoration: BoxDecoration(color: kOchaGreen, borderRadius: BorderRadius.circular(10)),
                    child: Text('$badge', style: const TextStyle(color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold)),
                  ),
                ),
            ],
          ),
        ),
      ),
    );
  }}

  // ==================== MÀN HÌNH QUẢN LÝ MÓN HÀNG & DANH MỤC SIÊU TỐC (SPREADSHEET + GOOGLE SHEETS) ====================
  Widget _buildSmartMenuSpreadsheetScreen() {{
    String filterCat = 'Tất cả';
    String searchKw = '';

    return StatefulBuilder(
      builder: (context, setSheetSt) {{
        final filteredList = allProducts.where((p) {{
          final bool matchCat = filterCat == 'Tất cả' || p['bottom_tab'] == filterCat;
          final bool matchKw = searchKw.isEmpty ||
              removeDiacritics(p['name'].toString()).contains(removeDiacritics(searchKw)) ||
              removeDiacritics(p['short_code'].toString()).contains(removeDiacritics(searchKw));
          return matchCat && matchKw;
        }}).toList();

        return Scaffold(
          backgroundColor: kOchaBgGrey,
          appBar: AppBar(
            backgroundColor: Colors.white,
            foregroundColor: Colors.black87,
            elevation: 1,
            leading: IconButton(
              icon: const Icon(Icons.arrow_back),
              onPressed: () =""" + f"""> setState(() =""" + f"""> fullScreenView = 'NONE'),
            ),
            title: const Text('Quản lý Thực đơn & Danh mục Thông minh (v11.0)', style: TextStyle(fontWeight: FontWeight.bold, fontSize: 17)),
            actions: [
              OutlinedButton.icon(
                style: OutlinedButton.styleFrom(foregroundColor: kOchaActionGreen, side: const BorderSide(color: kOchaGreen)),
                icon: const Icon(Icons.category_outlined, size: 18),
                label: const Text('Quản lý Danh mục (Tabs)'),
                onPressed: () async {{
                  await _showManageCategoriesDialog();
                  setSheetSt(() {{}});
                }},
              ),
              const SizedBox(width: 8),
              ElevatedButton.icon(
                style: ElevatedButton.styleFrom(backgroundColor: const Color(0xFF1565C0)),
                icon: const Icon(Icons.cloud_upload_outlined, size: 18),
                label: const Text('Export lên Google Sheets'),
                onPressed: _exportMenuToGoogleSheets,
              ),
              const SizedBox(width: 8),
              ElevatedButton.icon(
                style: ElevatedButton.styleFrom(backgroundColor: const Color(0xFF00897B)),
                icon: const Icon(Icons.cloud_download_outlined, size: 18),
                label: const Text('Import từ Google Sheets'),
                onPressed: () async {{
                  await _importMenuFromGoogleSheets();
                  setSheetSt(() {{}});
                }},
              ),
              const SizedBox(width: 8),
              PopupMenuButton{LT}String{GT}(
                tooltip: 'Xuất / Nhập bằng file CSV nội bộ',
                onSelected: (v) async {{
                  if (v == 'EXP_CSV') await _exportMenuToCsvFile();
                  if (v == 'IMP_CSV') {{
                    await _importMenuFromCsvFile();
                    setSheetSt(() {{}});
                  }}
                }},
                itemBuilder: (_) =""" + f"""> const [
                  PopupMenuItem(value: 'EXP_CSV', child: Text('📤 Xuất Thực đơn ra file CSV (Máy/USB)')),
                  PopupMenuItem(value: 'IMP_CSV', child: Text('📥 Nhập Thực đơn từ file CSV (Máy/USB)')),
                ],
                child: const Padding(
                  padding: EdgeInsets.symmetric(horizontal: 10),
                  child: Icon(Icons.more_vert),
                ),
              ),
            ],
          ),
          body: Column(
            children: [
              // Thanh lọc Danh mục + Tìm kiếm không dấu + Nút Thêm món mới
              Container(
                color: Colors.white,
                padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 10),
                child: Row(
                  children: [
                    Expanded(
                      child: SizedBox(
                        height: 38,
                        child: ListView(
                          scrollDirection: Axis.horizontal,
                          children: [
                            ...['Tất cả', ...allCategories.map((c) =""" + f"""> c['name'].toString())].map((cName) {{
                              final bool sel = filterCat == cName;
                              return Padding(
                                padding: const EdgeInsets.only(right: 8),
                                child: ChoiceChip(
                                  label: Text(cName, style: TextStyle(fontWeight: FontWeight.bold, color: sel ? Colors.white : Colors.black87)),
                                  selected: sel,
                                  selectedColor: kOchaGreen,
                                  onSelected: (_) =""" + f"""> setSheetSt(() =""" + f"""> filterCat = cName),
                                ),
                              );
                            }}),
                          ],
                        ),
                      ),
                    ),
                    SizedBox(
                      width: 240,
                      height: 38,
                      child: TextField(
                        decoration: const InputDecoration(
                          prefixIcon: Icon(Icons.search, size: 18),
                          hintText: 'Tìm nhanh (gõ không dấu)...',
                          isDense: true,
                          contentPadding: EdgeInsets.symmetric(vertical: 8),
                          border: OutlineInputBorder(),
                        ),
                        onChanged: (v) =""" + f"""> setSheetSt(() =""" + f"""> searchKw = v),
                      ),
                    ),
                    const SizedBox(width: 10),
                    ElevatedButton.icon(
                      style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen, padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 10)),
                      icon: const Icon(Icons.add),
                      label: const Text('+ Thêm Món Mới', style: TextStyle(fontWeight: FontWeight.bold)),
                      onPressed: () async {{
                        _showEditDishDialog(defaultTab: filterCat == 'Tất cả' ? 'CHUẨN' : filterCat);
                      }},
                    ),
                  ],
                ),
              ),
              const Divider(height: 1),

              // Bảng chỉnh sửa nhanh trực tiếp (Inline Spreadsheet View)
              Expanded(
                child: ListView.separated(
                  padding: const EdgeInsets.all(14),
                  itemCount: filteredList.length,
                  separatorBuilder: (_, __) =""" + f"""> const SizedBox(height: 6),
                  itemBuilder: (context, idx) {{
                    final item = filteredList[idx];
                    final bool inStock = (item['in_stock'] ?? 1) == 1;
                    final String imgPath = (item['image_path'] ?? '').toString();

                    return Card(
                      margin: EdgeInsets.zero,
                      elevation: 0.5,
                      child: Padding(
                        padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 6),
                        child: Row(
                          children: [
                            // Ảnh hoặc Mã viết tắt
                            InkWell(
                              onTap: () =""" + f"""> _showEditDishDialog(product: item),
                              child: SizedBox(
                                width: 54,
                                height: 46,
                                child: _buildOchaTileTopGraphic(imgPath, (item['short_code'] ?? 'BT').toString()),
                              ),
                            ),
                            const SizedBox(width: 12),
                            // Tên món & Danh mục
                            Expanded(
                              flex: 3,
                              child: Column(
                                crossAxisAlignment: CrossAxisAlignment.start,
                                children: [
                                  Text(item['name'].toString(), style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 16)),
                                  Text('Danh mục: ${{item['bottom_tab']}}  •  Mã ô: ${{item['short_code']}}', style: const TextStyle(color: Colors.black54, fontSize: 12)),
                                ],
                              ),
                            ),
                            // Nút chỉnh giá nhanh (-5k, Sửa số trực tiếp, +5k)
                            Row(
                              children: [
                                IconButton(
                                  tooltip: 'Giảm 5,000đ',
                                  icon: const Icon(Icons.remove_circle_outline, color: Colors.blueGrey),
                                  onPressed: () async {{
                                    final int newP = math.max(0, (item['price'] as int) - 5000);
                                    final db = await LocalDb.db;
                                    await db.update('products', {{'price': newP}}, where: 'id = ?', whereArgs: [item['id']]);
                                    await _loadData();
                                    setSheetSt(() {{}});
                                  }},
                                ),
                                InkWell(
                                  onTap: () =""" + f"""> _showEditDishDialog(product: item),
                                  child: Container(
                                    padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 6),
                                    decoration: BoxDecoration(
                                      color: Colors.grey.shade100,
                                      borderRadius: BorderRadius.circular(6),
                                      border: Border.all(color: Colors.grey.shade300),
                                    ),
                                    child: Text('${{fmt.format(item['price'])}}₫', style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 16, color: kOchaActionGreen)),
                                  ),
                                ),
                                IconButton(
                                  tooltip: 'Tăng 5,000đ',
                                  icon: const Icon(Icons.add_circle_outline, color: kOchaGreen),
                                  onPressed: () async {{
                                    final int newP = (item['price'] as int) + 5000;
                                    final db = await LocalDb.db;
                                    await db.update('products', {{'price': newP}}, where: 'id = ?', whereArgs: [item['id']]);
                                    await _loadData();
                                    setSheetSt(() {{}});
                                  }},
                                ),
                              ],
                            ),
                            const SizedBox(width: 14),
                            // Công tắc Còn hàng / Hết hàng
                            Column(
                              children: [
                                Switch(
                                  value: inStock,
                                  activeColor: kOchaGreen,
                                  onChanged: (val) async {{
                                    final db = await LocalDb.db;
                                    await db.update('products', {{'in_stock': val ? 1 : 0}}, where: 'id = ?', whereArgs: [item['id']]);
                                    await _loadData();
                                    setSheetSt(() {{}});
                                  }},
                                ),
                                Text(inStock ? 'Còn hàng' : 'HẾT HÀNG', style: TextStyle(fontSize: 10, fontWeight: FontWeight.bold, color: inStock ? kOchaGreen : Colors.red)),
                              ],
                            ),
                            const SizedBox(width: 12),
                            // Nút Nhân bản món (Clone) siêu tiện
                            OutlinedButton.icon(
                              icon: const Icon(Icons.copy, size: 15),
                              label: const Text('Nhân bản'),
                              onPressed: () async {{
                                final db = await LocalDb.db;
                                await db.insert('products', {{
                                  'bottom_tab': item['bottom_tab'],
                                  'short_code': item['short_code'],
                                  'name': '${{item['name']}} (Mới)',
                                  'price': item['price'],
                                  'image_path': item['image_path'] ?? '',
                                  'in_stock': 1,
                                }});
                                await _loadData();
                                setSheetSt(() {{}});
                              }},
                            ),
                            const SizedBox(width: 6),
                            IconButton(
                              tooltip: 'Sửa chi tiết / Đổi ảnh',
                              icon: const Icon(Icons.edit, color: Colors.blue),
                              onPressed: () =""" + f"""> _showEditDishDialog(product: item),
                            ),
                            IconButton(
                              tooltip: 'Xóa món',
                              icon: const Icon(Icons.delete_outline, color: Colors.red),
                              onPressed: () async {{
                                final db = await LocalDb.db;
                                await db.delete('products', where: 'id = ?', whereArgs: [item['id']]);
                                await _loadData();
                                setSheetSt(() {{}});
                              }},
                            ),
                          ],
                        ),
                      ),
                    );
                  }},
                ),
              ),
            ],
          ),
        );
      }},
    );
  }}

  // Hộp thoại Thêm / Đổi tên / Xóa Danh mục (Tabs dưới đáy)
  Future{LT}void{GT} _showManageCategoriesDialog() async {{
    final nameCtrl = TextEditingController();
    await showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setCatSt) =""" + f"""> AlertDialog(
          title: const Text('Quản lý Danh mục Thực đơn (Thanh dưới đáy)'),
          content: SizedBox(
            width: 500,
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                Row(
                  children: [
                    Expanded(
                      child: TextField(
                        controller: nameCtrl,
                        decoration: const InputDecoration(
                          labelText: 'Tên danh mục mới (VD: CHÈ, ĐỒ CHAY, COMBO...)',
                          border: OutlineInputBorder(),
                          isDense: true,
                        ),
                      ),
                    ),
                    const SizedBox(width: 10),
                    ElevatedButton.icon(
                      style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen, padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 13)),
                      icon: const Icon(Icons.add),
                      label: const Text('Thêm Tab'),
                      onPressed: () async {{
                        final n = nameCtrl.text.trim();
                        if (n.isEmpty) return;
                        final db = await LocalDb.db;
                        await db.insert('categories', {{'name': n, 'icon_code': 'food', 'sort_order': allCategories.length + 1}}, conflictAlgorithm: ConflictAlgorithm.ignore);
                        nameCtrl.clear();
                        await _loadData();
                        setCatSt(() {{}});
                      }},
                    ),
                  ],
                ),
                const SizedBox(height: 14),
                SizedBox(
                  height: 240,
                  child: ListView.separated(
                    itemCount: allCategories.length,
                    separatorBuilder: (_, __) =""" + f"""> const Divider(height: 1),
                    itemBuilder: (context, i) {{
                      final c = allCategories[i];
                      final String cName = c['name'].toString();
                      final int cnt = allProducts.where((p) =""" + f"""> p['bottom_tab'] == cName).length;
                      return ListTile(
                        leading: const Icon(Icons.folder_special, color: kOchaGreen),
                        title: Text(cName, style: const TextStyle(fontWeight: FontWeight.bold)),
                        subtitle: Text('$cnt món hàng'),
                        trailing: IconButton(
                          icon: const Icon(Icons.delete_outline, color: Colors.red),
                          onPressed: allCategories.length {LT}= 1
                              ? null
                              : () async {{
                                  final db = await LocalDb.db;
                                  await db.delete('categories', where: 'id = ?', whereArgs: [c['id']]);
                                  await _loadData();
                                  setCatSt(() {{}});
                                }},
                        ),
                      );
                    }},
                  ),
                ),
              ],
            ),
          ),
          actions: [
            TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Đóng')),
          ],
        ),
      ),
    );
  }}

  // ==================== MÀN HÌNH HÓA ĐƠN MỞ & BÀN ====================
  Widget _buildOpenBillsAndTablesView() {{
    final topTabs = ['Hóa đơn', ...allAreaNames];

    return Column(
      children: [
        Container(
          height: 54,
          color: const Color(0xFFF7F8FA),
          padding: const EdgeInsets.symmetric(horizontal: 16),
          child: Row(
            children: [
              Expanded(
                child: ListView.builder(
                  scrollDirection: Axis.horizontal,
                  itemCount: topTabs.length,
                  itemBuilder: (context, i) {{
                    final tab = topTabs[i];
                    final bool active = currentTopAreaTab == tab;
                    return InkWell(
                      onTap: () =""" + f"""> setState(() {{
                        currentTopAreaTab = tab;
                        isDeleteBillMode = false;
                        isEditTableMode = false;
                      }}),
                      child: Container(
                        padding: const EdgeInsets.symmetric(horizontal: 24),
                        alignment: Alignment.center,
                        decoration: BoxDecoration(
                          border: Border(
                            bottom: BorderSide(color: active ? kOchaGreen : Colors.transparent, width: 2.8),
                          ),
                        ),
                        child: Text(
                          tab,
                          style: TextStyle(
                            fontSize: 16,
                            fontWeight: active ? FontWeight.w600 : FontWeight.w400,
                            color: active ? const Color(0xFF333333) : const Color(0xFF666666),
                          ),
                        ),
                      ),
                    );
                  }},
                ),
              ),

              OutlinedButton.icon(
                style: OutlinedButton.styleFrom(
                  foregroundColor: kOchaActionGreen,
                  side: const BorderSide(color: kOchaGreen),
                  padding: const EdgeInsets.symmetric(horizontal: 10, vertical: 6),
                ),
                icon: const Icon(Icons.layers_outlined, size: 16),
                label: const Text('Tạo/Xóa Khu vực', style: TextStyle(fontSize: 12)),
                onPressed: _showManageAreasModal,
              ),
              const SizedBox(width: 8),

              if (currentTopAreaTab != 'Hóa đơn') ...[
                ElevatedButton.icon(
                  style: ElevatedButton.styleFrom(
                    backgroundColor: kOchaGreen,
                    elevation: 0,
                    padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 8),
                  ),
                  icon: const Icon(Icons.add, size: 16),
                  label: Text('Tạo bàn ($currentTopAreaTab)', style: const TextStyle(fontSize: 12)),
                  onPressed: () =""" + f"""> _showCreateOrEditTableDialog(defaultArea: currentTopAreaTab),
                ),
                const SizedBox(width: 4),
                IconButton(
                  tooltip: 'Bật/Tắt chế độ Xóa hoặc Sửa bàn',
                  icon: Icon(
                    isEditTableMode ? Icons.check_circle : Icons.edit_note,
                    color: isEditTableMode ? kOchaOccupiedRed : Colors.black54,
                  ),
                  onPressed: () =""" + f"""> setState(() =""" + f"""> isEditTableMode = !isEditTableMode),
                ),
              ] else ...[
                if (isDeleteBillMode)
                  TextButton(
                    onPressed: () =""" + f"""> setState(() =""" + f"""> isDeleteBillMode = false),
                    child: const Text('Hoàn tất', style: TextStyle(color: kOchaGreen, fontWeight: FontWeight.bold, fontSize: 16)),
                  )
                else
                  Row(
                    children: [
                      IconButton(
                        tooltip: 'Quản lý Thực đơn & Google Sheets',
                        icon: const Icon(Icons.grid_view_rounded, color: kOchaActionGreen, size: 22),
                        onPressed: () async {{
                          if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Quản lý Thực đơn')) {{
                            setState(() =""" + f"""> fullScreenView = 'SMART_MENU');
                          }}
                        }},
                      ),
                      IconButton(
                        tooltip: 'Cài đặt Âm thanh Ting-Ting',
                        icon: Icon(
                          soundMode == 'OFF' ? Icons.volume_off : Icons.volume_up,
                          color: soundMode == 'OFF' ? Colors.redAccent : kOchaGreen,
                          size: 22,
                        ),
                        onPressed: () =""" + f"""> setState(() {{
                          settingsSubTab = 'SOUND';
                          fullScreenView = 'CAI_DAT';
                        }}),
                      ),
                      IconButton(
                        tooltip: 'Chọn xóa từng hóa đơn',
                        icon: const Icon(Icons.delete_outline, color: Colors.black45, size: 22),
                        onPressed: () async {{
                          if (await _verifyPermissionOrAskAdminPin(canDeleteOrder, 'Xóa hóa đơn đang mở')) {{
                            setState(() =""" + f"""> isDeleteBillMode = true);
                          }}
                        }},
                      ),
                    ],
                  ),
              ],
            ],
          ),
        ),
        const Divider(height: 1, color: Color(0xFFDADCE0)),

        Expanded(
          child: currentTopAreaTab != 'Hóa đơn'
              ? _buildOchaVisualMapForArea(currentTopAreaTab, isInsideDialog: false)
              : GridView.builder(
                  padding: const EdgeInsets.all(16),
                  gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
                    crossAxisCount: 4,
                    childAspectRatio: 1.56,
                    crossAxisSpacing: 14,
                    mainAxisSpacing: 14,
                  ),
                  itemCount: openOrders.length + 1,
                  itemBuilder: (context, idx) {{
                    if (idx == 0) {{
                      return InkWell(
                        onTap: () {{
                          if (isDeleteBillMode) return;
                          _startNewOrder();
                        }},
                        child: Container(
                          decoration: BoxDecoration(
                            color: const Color(0xFFECEDEF),
                            border: Border.all(color: const Color(0xFFD5D7DB)),
                            borderRadius: BorderRadius.circular(2),
                          ),
                          child: Column(
                            crossAxisAlignment: CrossAxisAlignment.start,
                            children: const [
                              Padding(
                                padding: EdgeInsets.fromLTRB(14, 10, 14, 10),
                                child: Text('Hóa đơn mới', style: TextStyle(color: Color(0xFF555555), fontSize: 14, fontWeight: FontWeight.w500)),
                              ),
                              Divider(height: 1, color: Color(0xFFD5D7DB)),
                              Expanded(child: Center(child: Icon(Icons.add, size: 34, color: Color(0xFF666666)))),
                            ],
                          ),
                        ),
                      );
                    }}

                    final ord = openOrders[idx - 1];
                    final bool isTakeaway = ord['order_type'] == 'Mang về' || ord['table_name'].toString().startsWith('Số ');
                    final String tkCode = (ord['takeaway_code'] ?? 'T11001').toString();
                    final Color topAccentColor = isTakeaway ? kOchaTakeawayBlue : kOchaGreen;

                    return Container(
                      decoration: BoxDecoration(
                        color: Colors.white,
                        border: Border.all(color: const Color(0xFFDADCE0)),
                        borderRadius: BorderRadius.circular(2),
                      ),
                      child: Column(
                        children: [
                          Container(height: 2.5, color: topAccentColor),
                          SizedBox(
                            height: 38,
                            child: Padding(
                              padding: const EdgeInsets.only(left: 10, right: 12),
                              child: Row(
                                crossAxisAlignment: CrossAxisAlignment.start,
                                children: [
                                  ClipPath(
                                    clipper: OchaRibbonClipper(),
                                    child: Container(
                                      width: 24,
                                      height: 32,
                                      color: topAccentColor,
                                      padding: const EdgeInsets.only(bottom: 4),
                                      alignment: Alignment.center,
                                      child: Icon(
                                        isTakeaway ? Icons.shopping_bag_outlined : Icons.room_service_outlined,
                                        color: Colors.white,
                                        size: 14,
                                      ),
                                    ),
                                  ),
                                  const SizedBox(width: 8),
                                  Expanded(
                                    child: Align(
                                      alignment: Alignment.centerLeft,
                                      child: Text(
                                        ord['table_name'].toString(),
                                        maxLines: 1,
                                        overflow: TextOverflow.ellipsis,
                                        style: const TextStyle(fontWeight: FontWeight.w500, fontSize: 14, color: Color(0xFF333333)),
                                      ),
                                    ),
                                  ),
                                  Align(
                                    alignment: Alignment.centerRight,
                                    child: isTakeaway
                                        ? Text(tkCode, style: const TextStyle(fontSize: 13, color: Color(0xFF555555)))
                                        : Row(
                                            mainAxisSize: MainAxisSize.min,
                                            children: [
                                              Container(
                                                padding: const EdgeInsets.all(2),
                                                decoration: BoxDecoration(color: const Color(0xFFEE6B3B), borderRadius: BorderRadius.circular(2)),
                                                child: const Icon(Icons.receipt, size: 10, color: Colors.white),
                                              ),
                                              const SizedBox(width: 5),
                                              const Icon(Icons.person_outline, size: 16, color: Color(0xFF555555)),
                                              const SizedBox(width: 2),
                                              Text('${{ord['guest_count'] ?? 1}}', style: const TextStyle(fontSize: 13, color: Color(0xFF333333))),
                                            ],
                                          ),
                                  ),
                                ],
                              ),
                            ),
                          ),
                          const Divider(height: 1, color: Color(0xFFEEEEEE)),
                          Expanded(
                            child: InkWell(
                              onTap: () {{
                                if (!isDeleteBillMode) _openExistingOrder(ord);
                              }},
                              child: Padding(
                                padding: const EdgeInsets.symmetric(horizontal: 12),
                                child: Row(
                                  mainAxisAlignment: MainAxisAlignment.end,
                                  children: [
                                    Text('${{fmt.format(ord['total'])}}₫', style: const TextStyle(fontWeight: FontWeight.w600, fontSize: 16, color: Color(0xFF333333))),
                                  ],
                                ),
                              ),
                            ),
                          ),
                          const Divider(height: 1, color: Color(0xFFEEEEEE)),
                          SizedBox(
                            height: 38,
                            child: isDeleteBillMode
                                ? InkWell(
                                    onTap: () async {{
                                      final db = await LocalDb.db;
                                      await db.delete('orders', where: 'id = ?', whereArgs: [ord['id']]);
                                      await _loadData();
                                    }},
                                    child: const Center(
                                      child: Text('Xóa', style: TextStyle(color: kOchaOccupiedRed, fontWeight: FontWeight.w600, fontSize: 14)),
                                    ),
                                  )
                                : Row(
                                    children: [
                                      Expanded(
                                        child: InkWell(
                                          onTap: () =""" + f"""> _showPaymentCheckoutModal(directOrder: ord),
                                          child: const Center(
                                            child: Text('Thanh toán', style: TextStyle(color: kOchaActionGreen, fontWeight: FontWeight.w500, fontSize: 14)),
                                          ),
                                        ),
                                      ),
                                      Container(width: 1, height: 20, color: const Color(0xFFE0E0E0)),
                                      Expanded(
                                        child: InkWell(
                                          onTap: () =""" + f"""> _openExistingOrder(ord),
                                          child: const Center(
                                            child: Text('Xem hóa đơn', style: TextStyle(color: kOchaActionGreen, fontWeight: FontWeight.w500, fontSize: 14)),
                                          ),
                                        ),
                                      ),
                                    ],
                                  ),
                          ),
                        ],
                      ),
                    );
                  }},
                ),
        ),
      ],
    );
  }}

  Widget _buildOchaVisualMapForArea(String areaName, {{required bool isInsideDialog}}) {{
    final areaTables = allDiningTables.where((t) =""" + f"""> t['area'] == areaName).toList();

    if (areaTables.isEmpty) {{
      return Container(
        margin: const EdgeInsets.all(20),
        decoration: BoxDecoration(color: Colors.white, border: Border.all(color: const Color(0xFFE0E2E6))),
        alignment: Alignment.center,
        child: Column(
          mainAxisSize: MainAxisSize.min,
          children: [
            const Icon(Icons.table_restaurant_outlined, size: 54, color: Colors.grey),
            const SizedBox(height: 10),
            Text('Phân khu [$areaName] hiện chưa có bàn nào', style: const TextStyle(color: Colors.grey, fontSize: 16)),
            const SizedBox(height: 12),
            ElevatedButton.icon(
              style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen),
              icon: const Icon(Icons.add),
              label: Text('Tạo bàn đầu tiên cho [$areaName]'),
              onPressed: () =""" + f"""> _showCreateOrEditTableDialog(defaultArea: areaName),
            ),
          ],
        ),
      );
    }}

    return Container(
      margin: EdgeInsets.all(isInsideDialog ? 4 : 20),
      padding: EdgeInsets.symmetric(horizontal: isInsideDialog ? 12 : 36, vertical: isInsideDialog ? 8 : 18),
      decoration: BoxDecoration(color: Colors.white, border: Border.all(color: const Color(0xFFE0E2E6))),
      child: GridView.builder(
        gridDelegate: SliverGridDelegateWithFixedCrossAxisCount(
          crossAxisCount: 8,
          childAspectRatio: isInsideDialog ? 1.05 : 1.15,
          crossAxisSpacing: isInsideDialog ? 8 : 18,
          mainAxisSpacing: isInsideDialog ? 8 : 14,
        ),
        itemCount: areaTables.length,
        itemBuilder: (context, idx) {{
          final tObj = areaTables[idx];
          final String tCode = (tObj['table_code'] ?? '${{idx + 1}}').toString();
          final String tName = tObj['table_name'].toString();

          dynamic activeOrd;
          for (var o in openOrders) {{
            if (o['table_name'] == tName) {{
              activeOrd = o;
              break;
            }}
          }}
          final bool occupied = activeOrd != null;
          final bool prebilled = occupied && ((activeOrd['prebill_printed'] ?? 0) == 1);
          final Color tableColor = occupied ? kOchaOccupiedRed : kOchaGreen;

          return Stack(
            children: [
              InkWell(
                onTap: () {{
                  if (isEditTableMode && !isInsideDialog) {{
                    _showCreateOrEditTableDialog(tableObj: tObj);
                    return;
                  }}
                  if (isInsideDialog) {{
                    setState(() {{
                      activeOrderType = 'Dùng tại quán';
                      activeTableName = tName;
                      activeArea = areaName;
                    }});
                    Navigator.pop(context, tName);
                    return;
                  }}
                  if (occupied) {{
                    _openExistingOrder(activeOrd);
                  }} else {{
                    _startNewOrder(tableName: tName, area: areaName, orderType: 'Dùng tại quán');
                  }}
                }},
                onLongPress: () =""" + f"""> _showCreateOrEditTableDialog(tableObj: tObj),
                child: CustomPaint(
                  painter: OchaTableShapePainter(
                    color: tableColor,
                    hasOrder: occupied,
                    showBillBadge: occupied && (prebilled || tCode == '30'),
                  ),
                  child: Center(
                    child: Padding(
                      padding: const EdgeInsets.all(6),
                      child: Text(
                        tCode,
                        maxLines: 1,
                        overflow: TextOverflow.ellipsis,
                        style: TextStyle(fontSize: isInsideDialog ? 14 : 15, fontWeight: FontWeight.w500, color: const Color(0xFF666666)),
                      ),
                    ),
                  ),
                ),
              ),
              if (isEditTableMode && !isInsideDialog)
                Positioned(
                  top: 0,
                  right: 0,
                  child: InkWell(
                    onTap: () async {{
                      final db = await LocalDb.db;
                      await db.delete('dining_tables', where: 'id = ?', whereArgs: [tObj['id']]);
                      await _loadData();
                    }},
                    child: Container(
                      width: 22,
                      height: 22,
                      decoration: const BoxDecoration(color: kOchaOccupiedRed, shape: BoxShape.circle),
                      child: const Icon(Icons.close, size: 14, color: Colors.white),
                    ),
                  ),
                ),
            ],
          );
        }},
      ),
    );
  }}

  // ==================== LƯỚI THỰC ĐƠN 5x5 + TÌM KIẾM KHÔNG DẤU SIÊU TỐC ====================
  Widget _buildProductGrid5x5() {{
    final tabProducts = allProducts.where((p) {{
      if (posFastSearch.isNotEmpty) {{
        final kw = removeDiacritics(posFastSearch);
        return removeDiacritics(p['name'].toString()).contains(kw) ||
            removeDiacritics(p['short_code'].toString()).contains(kw);
      }}
      return p['bottom_tab'] == currentBottomTab;
    }}).toList();

    return Column(
      children: [
        Container(
          height: 46,
          color: Colors.white,
          padding: const EdgeInsets.symmetric(horizontal: 10),
          child: Row(
            children: [
              SizedBox(
                width: 220,
                height: 34,
                child: TextField(
                  decoration: InputDecoration(
                    prefixIcon: const Icon(Icons.search, size: 17),
                    hintText: 'Tìm nhanh (VD: db ko cha)...',
                    isDense: true,
                    contentPadding: const EdgeInsets.symmetric(vertical: 6),
                    border: OutlineInputBorder(borderRadius: BorderRadius.circular(4)),
                  ),
                  onChanged: (v) =""" + f"""> setState(() =""" + f"""> posFastSearch = v),
                ),
              ),
              const Spacer(),
              TextButton.icon(
                icon: const Icon(Icons.table_chart_outlined, size: 17, color: kOchaActionGreen),
                label: const Text('Bảng Quản lý Món & Sheets', style: TextStyle(color: kOchaActionGreen, fontWeight: FontWeight.bold)),
                onPressed: () async {{
                  if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Quản lý Thực đơn')) {{
                    setState(() =""" + f"""> fullScreenView = 'SMART_MENU');
                  }}
                }},
              ),
              TextButton.icon(
                icon: const Icon(Icons.add_circle_outline, size: 17),
                label: const Text('Thêm món'),
                onPressed: () async {{
                  if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Thêm món')) {{
                    _showEditDishDialog(defaultTab: currentBottomTab);
                  }}
                }},
              ),
              OutlinedButton.icon(
                style: OutlinedButton.styleFrom(
                  foregroundColor: isEditGridMode ? Colors.red : Colors.black87,
                  padding: const EdgeInsets.symmetric(horizontal: 8),
                ),
                icon: Icon(isEditGridMode ? Icons.check : Icons.edit, size: 15),
                label: Text(isEditGridMode ? 'Xong' : 'Sửa ô', style: const TextStyle(fontSize: 12)),
                onPressed: () async {{
                  if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Chỉnh sửa Thực đơn')) {{
                    setState(() =""" + f"""> isEditGridMode = !isEditGridMode);
                  }}
                }},
              ),
            ],
          ),
        ),
        const Divider(height: 1),

        Expanded(
          child: GridView.builder(
            padding: const EdgeInsets.all(8),
            gridDelegate: const SliverGridDelegateWithFixedCrossAxisCount(
              crossAxisCount: 5,
              childAspectRatio: 1.18,
              crossAxisSpacing: 8,
              mainAxisSpacing: 8,
            ),
            itemCount: tabProducts.length,
            itemBuilder: (context, idx) {{
              final prod = tabProducts[idx];
              final String imgPath = (prod['image_path'] ?? '').toString();
              final String code = (prod['short_code'] ?? 'BT').toString();
              final bool inStock = (prod['in_stock'] ?? 1) == 1;

              return Stack(
                children: [
                  InkWell(
                    onTap: () {{
                      if (isEditGridMode) {{
                        _showEditDishDialog(product: prod);
                      }} else {{
                        _addProductToCart(prod);
                      }}
                    }},
                    onLongPress: () async {{
                      if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Chỉnh sửa món')) {{
                        _showEditDishDialog(product: prod);
                      }}
                    }},
                    child: Opacity(
                      opacity: inStock ? 1.0 : 0.45,
                      child: Container(
                        decoration: BoxDecoration(
                          color: Colors.white,
                          borderRadius: BorderRadius.circular(4),
                          border: Border.all(color: Colors.grey.shade300),
                        ),
                        child: Column(
                          crossAxisAlignment: CrossAxisAlignment.stretch,
                          children: [
                            Expanded(flex: 65, child: _buildOchaTileTopGraphic(imgPath, code)),
                            Expanded(
                              flex: 35,
                              child: Container(
                                color: Colors.white,
                                alignment: Alignment.center,
                                padding: const EdgeInsets.symmetric(horizontal: 4),
                                child: Text(
                                  prod['name'].toString(),
                                  maxLines: 1,
                                  overflow: TextOverflow.ellipsis,
                                  textAlign: TextAlign.center,
                                  style: const TextStyle(fontSize: 13, color: Colors.black87, fontWeight: FontWeight.w500),
                                ),
                              ),
                            ),
                          ],
                        ),
                      ),
                    ),
                  ),

                  if (!inStock)
                    Positioned(
                      top: 6,
                      right: 6,
                      child: Container(
                        padding: const EdgeInsets.symmetric(horizontal: 6, vertical: 2),
                        decoration: BoxDecoration(color: Colors.red.shade700, borderRadius: BorderRadius.circular(4)),
                        child: const Text('HẾT HÀNG', style: TextStyle(color: Colors.white, fontSize: 10, fontWeight: FontWeight.bold)),
                      ),
                    ),

                  if (isEditGridMode)
                    Positioned(
                      top: 3,
                      left: 3,
                      child: InkWell(
                        onTap: () async {{
                          final db = await LocalDb.db;
                          await db.delete('products', where: 'id = ?', whereArgs: [prod['id']]);
                          await _loadData();
                        }},
                        child: Container(
                          width: 22,
                          height: 22,
                          decoration: const BoxDecoration(color: Colors.redAccent, shape: BoxShape.circle),
                          child: const Icon(Icons.close, size: 14, color: Colors.white),
                        ),
                      ),
                    ),
                ],
              );
            }},
          ),
        ),
      ],
    );
  }}

  Widget _buildOchaTileTopGraphic(String imgPath, String shortCode) {{
    if (imgPath.isNotEmpty && File(imgPath).existsSync()) {{
      return Image.file(File(imgPath), fit: BoxFit.cover);
    }}
    return Container(
      color: const Color(0xFFD5D9DF),
      alignment: Alignment.center,
      child: Container(
        padding: const EdgeInsets.symmetric(horizontal: 8, vertical: 3),
        decoration: BoxDecoration(border: Border.all(color: Colors.white, width: 1.2)),
        child: Text(shortCode, style: const TextStyle(color: Colors.white, fontSize: 12, fontWeight: FontWeight.bold)),
      ),
    );
  }}

  Widget _buildLibraryLeftPane() {{
    final filtered = allProducts.where((p) {{
      if (librarySearch.isNotEmpty) {{
        return removeDiacritics(p['name'].toString()).contains(removeDiacritics(librarySearch));
      }}
      if (selectedLibrarySection == 'Tất cả mặt hàng' || selectedLibrarySection == 'Lá Chuối Quán' || selectedLibrarySection == 'Món ăn') return true;
      if (selectedLibrarySection == 'Bánh Cuốn') return p['bottom_tab'] == 'CHUẨN';
      if (selectedLibrarySection == 'Bánh Ít') return p['bottom_tab'] == 'BÁNH-BÚN';
      return true;
    }}).toList();

    return Column(
      children: [
        Container(
          height: 52,
          color: Colors.white,
          padding: const EdgeInsets.symmetric(horizontal: 16),
          child: Row(
            children: [
              if (selectedLibrarySection != 'ROOT')
                IconButton(
                  icon: const Icon(Icons.arrow_back),
                  onPressed: () =""" + f"""> setState(() =""" + f"""> selectedLibrarySection = 'ROOT'),
                ),
              const Expanded(child: Center(child: Text('Thư viện', style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)))),
              ElevatedButton.icon(
                style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen),
                icon: const Icon(Icons.grid_view, size: 16),
                label: const Text('Quản lý Món & Google Sheets'),
                onPressed: () async {{
                  if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Quản lý Thực đơn')) {{
                    setState(() =""" + f"""> fullScreenView = 'SMART_MENU');
                  }}
                }},
              ),
            ],
          ),
        ),
        const Divider(height: 1),
        Container(
          color: Colors.white,
          padding: const EdgeInsets.all(10),
          child: TextField(
            decoration: InputDecoration(
              prefixIcon: const Icon(Icons.search),
              hintText: 'Tìm mặt hàng, giảm giá và phụ phí',
              isDense: true,
              border: OutlineInputBorder(borderRadius: BorderRadius.circular(4)),
            ),
            onChanged: (v) =""" + f"""> setState(() =""" + f"""> librarySearch = v),
          ),
        ),
        const Divider(height: 1),
        Expanded(
          child: selectedLibrarySection == 'ROOT' && librarySearch.isEmpty
              ? ListView(
                  children: [
                    _buildLibraryRow(badge: '+S-', title: 'Thêm số tiền tuỳ chỉnh', hasArrow: false, onTap: _showCustomAmountDialog),
                    _buildLibraryRow(badge: '%', title: 'Giảm giá', onTap: _showDiscountDialog),
                    _buildLibraryRow(badge: '+', title: 'Phụ phí', onTap: _showSurchargeDialog),
                    _buildLibraryRow(badge: 'All', title: 'Tất cả mặt hàng', onTap: () =""" + f"""> setState(() =""" + f"""> selectedLibrarySection = 'Tất cả mặt hàng')),
                    _buildLibraryRow(badge: 'BÁ', title: 'Bánh Cuốn', onTap: () =""" + f"""> setState(() =""" + f"""> selectedLibrarySection = 'Bánh Cuốn')),
                    _buildLibraryRow(badge: 'BÁ', title: 'Bánh Ít', onTap: () =""" + f"""> setState(() =""" + f"""> selectedLibrarySection = 'Bánh Ít')),
                    _buildLibraryRow(badge: 'LÁ', title: 'Lá Chuối Quán', onTap: () =""" + f"""> setState(() =""" + f"""> selectedLibrarySection = 'Lá Chuối Quán')),
                    _buildLibraryRow(badge: 'MÓ', title: 'Món ăn (Sửa nhanh / Export / Import)', onTap: () async {{
                      if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Quản lý Thực đơn')) {{
                        setState(() =""" + f"""> fullScreenView = 'SMART_MENU');
                      }}
                    }}),
                  ],
                )
              : ListView.separated(
                  itemCount: filtered.length,
                  separatorBuilder: (_, __) =""" + f"""> const Divider(height: 1),
                  itemBuilder: (context, idx) {{
                    final p = filtered[idx];
                    final bool inStock = (p['in_stock'] ?? 1) == 1;
                    return ListTile(
                      tileColor: Colors.white,
                      leading: SizedBox(
                        width: 44,
                        height: 44,
                        child: _buildOchaTileTopGraphic((p['image_path'] ?? '').toString(), (p['short_code'] ?? 'MÓ').toString()),
                      ),
                      title: Text(p['name'].toString(), style: const TextStyle(fontWeight: FontWeight.bold)),
                      subtitle: Text('Tab: ${{p['bottom_tab']}}  •  ${{inStock ? "Còn hàng" : "HẾT HÀNG"}}'),
                      trailing: Row(
                        mainAxisSize: MainAxisSize.min,
                        children: [
                          Switch(
                            value: inStock,
                            activeColor: kOchaGreen,
                            onChanged: (val) async {{
                              final db = await LocalDb.db;
                              await db.update('products', {{'in_stock': val ? 1 : 0}}, where: 'id = ?', whereArgs: [p['id']]);
                              await _loadData();
                            }},
                          ),
                          Text('${{fmt.format(p['price'])}}₫', style: const TextStyle(fontWeight: FontWeight.bold, fontSize: 15)),
                          const SizedBox(width: 8),
                          IconButton(
                            icon: const Icon(Icons.edit_outlined, color: Colors.blue),
                            onPressed: () =""" + f"""> _showEditDishDialog(product: p),
                          ),
                        ],
                      ),
                      onTap: () =""" + f"""> _addProductToCart(p),
                    );
                  }},
                ),
        ),
      ],
    );
  }}

  Widget _buildLibraryRow({{required String badge, required String title, bool hasArrow = true, required VoidCallback onTap}}) {{
    return Column(
      children: [
        ListTile(
          tileColor: Colors.white,
          onTap: onTap,
          leading: Container(
            width: 42,
            height: 36,
            color: const Color(0xFFD5D9DF),
            alignment: Alignment.center,
            child: Container(
              padding: const EdgeInsets.symmetric(horizontal: 4, vertical: 1),
              decoration: BoxDecoration(border: Border.all(color: Colors.white)),
              child: Text(badge, style: const TextStyle(color: Colors.white, fontSize: 11, fontWeight: FontWeight.bold)),
            ),
          ),
          title: Text(title, style: const TextStyle(fontSize: 15)),
          trailing: hasArrow ? const Icon(Icons.chevron_right, color: Colors.grey) : null,
        ),
        const Divider(height: 1),
      ],
    );
  }}

  Widget _buildRightOrderPanel() {{
    final bool isUnselectedTable = activeTableName == 'Chọn bàn';

    return Container(
      color: Colors.white,
      child: Column(
        children: [
          Container(
            height: 52,
            padding: const EdgeInsets.symmetric(horizontal: 12),
            child: Row(
              children: [
                PopupMenuButton{LT}String{GT}(
                  onSelected: (val) {{
                    setState(() {{
                      activeOrderType = val;
                      if (val == 'Mang về') {{
                        final tk = _generateNextTakeawayInfo();
                        activeTableName = tk['name']!;
                        activeTakeawayCode = tk['code']!;
                        activeArea = 'Mang về';
                      }} else if (val == 'Dùng tại quán' && activeTableName.startsWith('Số ')) {{
                        activeTableName = 'Chọn bàn';
                        activeTakeawayCode = '';
                        activeArea = 'Bàn';
                      }}
                    }});
                  }},
                  itemBuilder: (_) =""" + f"""> const [
                    PopupMenuItem(value: 'Dùng tại quán', child: Text('🍽 Dùng tại quán')),
                    PopupMenuItem(value: 'Mang về', child: Text('🛍 Mang về')),
                  ],
                  child: Row(
                    children: [
                      Icon(
                        activeOrderType == 'Mang về' ? Icons.shopping_bag_outlined : Icons.room_service,
                        color: activeOrderType == 'Mang về' ? kOchaTakeawayBlue : kOchaGreen,
                        size: 20,
                      ),
                      const Icon(Icons.keyboard_arrow_down, size: 18, color: Colors.black54),
                    ],
                  ),
                ),
                const SizedBox(width: 6),
                Expanded(
                  child: InkWell(
                    onTap: () =""" + f"""> _showSelectTableVisualModal(forSavingNow: false),
                    child: Text(
                      activeTableName,
                      maxLines: 1,
                      overflow: TextOverflow.ellipsis,
                      style: TextStyle(
                        fontSize: 16,
                        fontWeight: FontWeight.bold,
                        color: isUnselectedTable ? const Color(0xFF4285F4) : Colors.black87,
                      ),
                    ),
                  ),
                ),
                InkWell(
                  onTap: _showGuestCountModal,
                  child: Row(
                    children: [
                      const Icon(Icons.person_outline, size: 18, color: Colors.black54),
                      const SizedBox(width: 4),
                      Text(activeGuestCount.toString().padLeft(2, '0'), style: const TextStyle(fontSize: 15, fontWeight: FontWeight.w600)),
                      const Icon(Icons.keyboard_arrow_down, size: 18, color: Colors.black54),
                    ],
                  ),
                ),
              ],
            ),
          ),
          const Divider(height: 1),

          if (cart.isNotEmpty)
            Container(
              width: double.infinity,
              padding: const EdgeInsets.symmetric(horizontal: 14, vertical: 8),
              color: const Color(0xFFF9FAFB),
              child: Text('🧾 1- $currentStaffName thêm món ($activeOrderType)', style: TextStyle(fontSize: 13, color: Colors.grey.shade600)),
            ),

          Expanded(
            child: cart.isEmpty
                ? const SizedBox()
                : ListView.separated(
                    itemCount: cart.length,
                    separatorBuilder: (_, __) =""" + f"""> const Divider(height: 1),
                    itemBuilder: (context, idx) {{
                      final it = cart[idx];
                      final int qty = it['qty'] as int;
                      final int unitPrice = (it['price'] as int) + ((it['modifier_price'] ?? 0) as int);
                      final int lineTotal = qty * unitPrice;
                      final String mods = (it['modifiers'] ?? '').toString();
                      final String note = (it['note'] ?? '').toString();

                      return ListTile(
                        dense: true,
                        onTap: () =""" + f"""> _showEditCartItemModifiersDialog(idx),
                        title: Text(qty == 1 ? it['name'].toString() : '$qty x  ${{it['name']}}', style: const TextStyle(fontSize: 15, fontWeight: FontWeight.w500)),
                        subtitle: (mods.isNotEmpty || note.isNotEmpty)
                            ? Text([if (mods.isNotEmpty) '+ $mods', if (note.isNotEmpty) 'Ghi chú: $note'].join(' | '), style: const TextStyle(color: Colors.deepOrange, fontSize: 12))
                            : null,
                        trailing: Text('${{fmt.format(lineTotal)}}₫', style: const TextStyle(fontSize: 15, fontWeight: FontWeight.bold)),
                      );
                    }},
                  ),
          ),

          Padding(
            padding: const EdgeInsets.symmetric(horizontal: 12, vertical: 6),
            child: Row(
              mainAxisAlignment: MainAxisAlignment.spaceBetween,
              children: [
                Row(
                  children: [
                    OutlinedButton.icon(
                      style: OutlinedButton.styleFrom(foregroundColor: Colors.grey.shade700, side: BorderSide(color: Colors.grey.shade300)),
                      icon: const Icon(Icons.delete_outline, size: 18),
                      label: const Text('Xóa tất cả'),
                      onPressed: cart.isEmpty
                          ? null
                          : () {{
                              _saveUndoSnapshot();
                              setState(() {{
                                cart.clear();
                                activeDiscount = 0;
                                activeSurcharge = 0;
                              }});
                            }},
                    ),
                    if (undoCartSnapshot.isNotEmpty) ...[
                      const SizedBox(width: 6),
                      TextButton.icon(
                        icon: const Icon(Icons.undo, size: 16),
                        label: const Text('Hoàn tác'),
                        onPressed: () {{
                          setState(() {{
                            cart = jsonDecode(jsonEncode(undoCartSnapshot));
                            undoCartSnapshot = [];
                          }});
                        }},
                      ),
                    ],
                  ],
                ),
                InkWell(
                  onTap: _showOrderMoreActionsPopup,
                  child: Container(
                    width: 38,
                    height: 38,
                    decoration: BoxDecoration(shape: BoxShape.circle, border: Border.all(color: Colors.grey.shade300)),
                    child: const Icon(Icons.more_horiz, color: Colors.black54),
                  ),
                ),
              ],
            ),
          ),
          const Divider(height: 1),

          Container(
            padding: const EdgeInsets.fromLTRB(14, 10, 14, 12),
            child: Column(
              children: [
                Row(
                  mainAxisAlignment: MainAxisAlignment.spaceBetween,
                  children: [
                    Column(
                      crossAxisAlignment: CrossAxisAlignment.start,
                      children: [
                        Text('Tổng ($cartItemCount mặt hàng)', style: const TextStyle(fontSize: 13, color: Colors.black54)),
                        const SizedBox(height: 2),
                        Text('${{fmt.format(cartTotal)}}₫', style: const TextStyle(fontSize: 22, fontWeight: FontWeight.bold, color: Colors.black87)),
                      ],
                    ),
                    InkWell(
                      onTap: () async {{
                        if (cart.isEmpty) return;
                        unawaited(OchaPrinter.printBill(
                          storeName: storeName,
                          storeAddress: storeAddress,
                          storePhone: storePhone,
                          receiptNo: activeReceiptNo,
                          orderType: activeOrderType,
                          tableName: activeTableName,
                          guestCount: activeGuestCount,
                          staffName: currentStaffName,
                          startTime: activeStartTime,
                          items: cart,
                          discount: activeDiscount,
                          surcharge: activeSurcharge,
                          total: cartTotal,
                          paymentMethod: 'Tiền mặt',
                          note: activeOrderNote,
                          printMode: 'PREBILL',
                          openDrawer: false,
                        ));
                        setState(() =""" + f"""> activePrebillPrinted = 1);
                      }},
                      child: Column(
                        children: const [
                          Icon(Icons.print_outlined, color: Color(0xFF4285F4), size: 26),
                          Text('In', style: TextStyle(fontSize: 12, color: Color(0xFF4285F4))),
                        ],
                      ),
                    ),
                  ],
                ),
                const SizedBox(height: 10),
                Row(
                  children: [
                    Expanded(
                      child: SizedBox(
                        height: 46,
                        child: ElevatedButton(
                          style: ElevatedButton.styleFrom(backgroundColor: const Color(0xFF4C8BF5), elevation: 0),
                          onPressed: _handlePressSaveButton,
                          child: const Text('Lưu', style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold, color: Colors.white)),
                        ),
                      ),
                    ),
                    const SizedBox(width: 10),
                    Expanded(
                      child: SizedBox(
                        height: 46,
                        child: ElevatedButton(
                          style: ElevatedButton.styleFrom(backgroundColor: cart.isEmpty ? Colors.grey.shade300 : kOchaGreen, elevation: 0),
                          onPressed: cart.isEmpty ? null : () =""" + f"""> _showPaymentCheckoutModal(),
                          child: const Text('Thanh toán', style: TextStyle(fontSize: 16, fontWeight: FontWeight.bold, color: Colors.white)),
                        ),
                      ),
                    ),
                  ],
                ),
              ],
            ),
          ),
        ],
      ),
    );
  }}

  // ==================== POPUP MENU (•••) GÓC DƯỚI TRÁI (CÓ KIỂM TRA PHÂN QUYỀN ADMIN/USER) ====================
  void _showNineDotsPopupMenu() {{
    showDialog(
      context: context,
      barrierColor: Colors.black26,
      builder: (context) =""" + f"""> Align(
        alignment: Alignment.bottomLeft,
        child: Padding(
          padding: const EdgeInsets.only(left: 12, bottom: 72),
          child: Material(
            color: const Color(0xFF2B3036).withOpacity(0.95),
            borderRadius: BorderRadius.circular(8),
            child: Container(
              width: 420,
              padding: const EdgeInsets.all(20),
              child: Column(
                mainAxisSize: MainAxisSize.min,
                children: [
                  Row(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Expanded(
                        child: Column(
                          mainAxisSize: MainAxisSize.min,
                          children: [
                            _buildDarkMenuItem(Icons.assignment_outlined, 'Hoạt động', kOchaGreen, () async {{
                              Navigator.pop(context);
                              if (await _verifyPermissionOrAskAdminPin(canViewActivity, 'Xem lịch sử Hoạt động')) {{
                                setState(() =""" + f"""> fullScreenView = 'HOAT_DONG');
                              }}
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.pie_chart_outline, 'Báo cáo', const Color(0xFF4285F4), () async {{
                              Navigator.pop(context);
                              if (await _verifyPermissionOrAskAdminPin(canViewReport, 'Xem Báo cáo Doanh thu')) {{
                                setState(() =""" + f"""> fullScreenView = 'BAO_CAO');
                              }}
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.grid_view_rounded, 'Quản lý Món', const Color(0xFFE57373), () async {{
                              Navigator.pop(context);
                              if (await _verifyPermissionOrAskAdminPin(canEditMenu, 'Quản lý Thực đơn')) {{
                                setState(() =""" + f"""> fullScreenView = 'SMART_MENU');
                              }}
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.monetization_on_outlined, 'Két tiền', const Color(0xFF5C6BC0), () async {{
                              Navigator.pop(context);
                              CashDrawerBellSound.playSound();
                              await SunmiPrinter.openDrawer();
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.settings_outlined, 'Cài đặt', const Color(0xFF5C6BC0), () async {{
                              Navigator.pop(context);
                              if (await _verifyPermissionOrAskAdminPin(isAdmin, 'Cài đặt hệ thống')) {{
                                setState(() {{
                                  settingsSubTab = 'USERS';
                                  fullScreenView = 'CAI_DAT';
                                }});
                              }}
                            }}),
                          ],
                        ),
                      ),
                      Expanded(
                        child: Column(
                          mainAxisSize: MainAxisSize.min,
                          children: [
                            _buildDarkMenuItem(Icons.people_outline, 'Phân quyền NV', const Color(0xFF7E57C2), () async {{
                              Navigator.pop(context);
                              if (await _verifyPermissionOrAskAdminPin(isAdmin, 'Quản lý Nhân viên & Phân quyền')) {{
                                setState(() {{
                                  settingsSubTab = 'USERS';
                                  fullScreenView = 'CAI_DAT';
                                }});
                              }}
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.volume_up, 'Âm thanh Ting', const Color(0xFFFF9800), () {{
                              Navigator.pop(context);
                              setState(() {{
                                settingsSubTab = 'SOUND';
                                fullScreenView = 'CAI_DAT';
                              }});
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.table_restaurant, 'Quản lý Bàn', kOchaOccupiedRed, () {{
                              Navigator.pop(context);
                              _showManageAreasModal();
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.lock_outline, 'Khóa màn hình', Colors.teal, () {{
                              Navigator.pop(context);
                              setState(() {{
                                enteredPin = '';
                                pinErrorMsg = '';
                                isLockedPinScreen = true;
                              }});
                            }}),
                            const SizedBox(height: 16),
                            _buildDarkMenuItem(Icons.info_outline, 'Phiên bản $APP_VERSION', const Color(0xFF00897B), () {{
                              Navigator.pop(context);
                              setState(() {{
                                settingsSubTab = 'VERSION';
                                fullScreenView = 'CAI_DAT';
                              }});
                            }}),
                          ],
                        ),
                      ),
                    ],
                  ),
                  const SizedBox(height: 14),
                  const Divider(color: Colors.white24, height: 1),
                  const SizedBox(height: 8),
                  Text('Đang trực: ${{currentUser['name']}} (${{currentUser['role']}}) • $APP_VERSION', style: const TextStyle(color: Colors.white70, fontSize: 12)),
                ],
              ),
            ),
          ),
        ),
      ),
    );
  }}

  Widget _buildDarkMenuItem(IconData icon, String title, Color bgColor, VoidCallback onTap) {{
    return InkWell(
      onTap: onTap,
      child: Row(
        children: [
          Container(
            width: 44,
            height: 44,
            decoration: BoxDecoration(color: bgColor, borderRadius: BorderRadius.circular(8)),
            child: Icon(icon, color: Colors.white, size: 24),
          ),
          const SizedBox(width: 12),
          Expanded(child: Text(title, style: const TextStyle(color: Colors.white, fontSize: 15, fontWeight: FontWeight.w500))),
        ],
      ),
    );
  }}

  // ==================== MÀN HÌNH CÀI ĐẶT (QUẢN LÝ USER & PHÂN QUYỀN XEM BÁO CÁO) ====================
  Widget _buildSettingsScreen() {{
    final webhookCtrl = TextEditingController(text: webhookUrl);
    final storeCtrl = TextEditingController(text: storeName);
    final addrCtrl = TextEditingController(text: storeAddress);
    final phoneCtrl = TextEditingController(text: storePhone);

    return Scaffold(
      backgroundColor: Colors.white,
      body: Row(
        children: [
          Expanded(
            flex: 34,
            child: Column(
              children: [
                Container(
                  height: 54,
                  padding: const EdgeInsets.symmetric(horizontal: 8),
                  child: Row(
                    children: [
                      IconButton(
                        icon: const Icon(Icons.arrow_back),
                        onPressed: () =""" + f"""> setState(() =""" + f"""> fullScreenView = 'NONE'),
                      ),
                      const Expanded(child: Center(child: Text('Cài đặt & Phân quyền', style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)))),
                    ],
                  ),
                ),
                const Divider(height: 1),
                Expanded(
                  child: ListView(
                    children: [
                      _buildSettingsNavTile('USERS', Icons.admin_panel_settings, 'Nhân viên & Phân quyền (Xem Báo cáo)'),
                      const Divider(height: 1),
                      _buildSettingsNavTile('SOUND', Icons.volume_up, 'Âm thanh Két tiền (Tắt / Tự tải file)'),
                      const Divider(height: 1),
                      _buildSettingsNavTile('STORE', Icons.storefront, 'Cửa hàng & Google Sheets Webhook'),
                      const Divider(height: 1),
                      _buildSettingsNavTile('TABLES', Icons.table_restaurant, 'Quản lý Bàn & Khu vực (Tạo/Xóa)'),
                      const Divider(height: 1),
                      _buildSettingsNavTile('VERSION', Icons.info_outline, 'Thông tin (Phiên bản $APP_VERSION)'),
                    ],
                  ),
                ),
                Padding(
                  padding: const EdgeInsets.all(16),
                  child: OutlinedButton(
                    style: OutlinedButton.styleFrom(minimumSize: const Size.fromHeight(48)),
                    onPressed: () =""" + f"""> setState(() =""" + f"""> fullScreenView = 'NONE'),
                    child: const Text('ĐÓNG CÀI ĐẶT'),
                  ),
                ),
              ],
            ),
          ),
          const VerticalDivider(width: 1),
          Expanded(
            flex: 66,
            child: Padding(
              padding: const EdgeInsets.all(28),
              child: settingsSubTab == 'USERS'
                  ? _buildUsersPermissionPanel()
                  : settingsSubTab == 'SOUND'
                      ? _buildSoundSettingsPanel()
                      : settingsSubTab == 'VERSION'
                          ? _buildVersionPanel()
                          : settingsSubTab == 'TABLES'
                              ? _buildTablesManagerSettingsPanel()
                              : ListView(
                                  children: [
                                    const Text('Thông tin Cửa hàng & Kết nối Google Sheets 2 Chiều', style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
                                    const SizedBox(height: 16),
                                    TextField(
                                      controller: storeCtrl,
                                      decoration: const InputDecoration(labelText: 'Tên hộ kinh doanh / Quán', border: OutlineInputBorder()),
                                      onChanged: (v) {{
                                        storeName = v;
                                        _saveSetting('store_name', v);
                                      }},
                                    ),
                                    const SizedBox(height: 12),
                                    TextField(
                                      controller: phoneCtrl,
                                      decoration: const InputDecoration(labelText: 'Số điện thoại', border: OutlineInputBorder()),
                                      onChanged: (v) {{
                                        storePhone = v;
                                        _saveSetting('store_phone', v);
                                      }},
                                    ),
                                    const SizedBox(height: 12),
                                    TextField(
                                      controller: addrCtrl,
                                      decoration: const InputDecoration(labelText: 'Địa chỉ', border: OutlineInputBorder()),
                                      onChanged: (v) {{
                                        storeAddress = v;
                                        _saveSetting('store_address', v);
                                      }},
                                    ),
                                    const SizedBox(height: 20),
                                    TextField(
                                      controller: webhookCtrl,
                                      decoration: const InputDecoration(
                                        labelText: 'Google Apps Script Webhook URL (Đồng bộ Đơn hàng & Export/Import Thực đơn)',
                                        hintText: 'https://script.google.com/macros/s/.../exec',
                                        border: OutlineInputBorder(),
                                      ),
                                      onChanged: (v) {{
                                        webhookUrl = v.trim();
                                        _saveSetting('webhook_url', webhookUrl);
                                      }},
                                    ),
                                  ],
                                ),
            ),
          ),
        ],
      ),
    );
  }}

  Widget _buildSettingsNavTile(String tabKey, IconData icon, String title) {{
    final bool sel = settingsSubTab == tabKey;
    return Container(
      color: sel ? kOchaGreen : Colors.transparent,
      child: ListTile(
        leading: Icon(icon, color: sel ? Colors.white : kOchaGreen),
        title: Text(title, style: TextStyle(color: sel ? Colors.white : Colors.black87, fontWeight: FontWeight.bold)),
        onTap: () =""" + f"""> setState(() =""" + f"""> settingsSubTab = tabKey),
      ),
    );
  }}

  // GIAO DIỆN QUẢN LÝ USER & PHÂN QUYỀN ADMIN / USER: CÓ CHO XEM BÁO CÁO HAY KHÔNG
  Widget _buildUsersPermissionPanel() {{
    return Column(
      crossAxisAlignment: CrossAxisAlignment.start,
      children: [
        Row(
          mainAxisAlignment: MainAxisAlignment.spaceBetween,
          children: [
            const Text('Quản lý Tài khoản & Phân quyền Nhân viên', style: TextStyle(fontSize: 19, fontWeight: FontWeight.bold)),
            ElevatedButton.icon(
              style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen, padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 10)),
              icon: const Icon(Icons.person_add),
              label: const Text('+ Thêm Nhân viên / User'),
              onPressed: () =""" + f"""> _showEditUserDialog(),
            ),
          ],
        ),
        const SizedBox(height: 8),
        SwitchListTile(
          contentPadding: EdgeInsets.zero,
          title: const Text('Yêu cầu nhập Mã PIN khi mở ứng dụng', style: TextStyle(fontWeight: FontWeight.bold)),
          subtitle: const Text('Khi bật, nhân viên phải bấm mã PIN 4 số của mình để vào ca bán hàng.'),
          value: requirePinOnStart,
          activeColor: kOchaGreen,
          onChanged: (val) {{
            setState(() =""" + f"""> requirePinOnStart = val);
            _saveSetting('require_pin_on_start', val ? '1' : '0');
          }},
        ),
        const Divider(),
        const Text('Bật/Tắt trực tiếp các quyền cho từng nhân viên bên dưới:', style: TextStyle(color: Colors.black54)),
        const SizedBox(height: 8),
        Expanded(
          child: ListView.separated(
            itemCount: allUsers.length,
            separatorBuilder: (_, __) =""" + f"""> const SizedBox(height: 10),
            itemBuilder: (context, idx) {{
              final u = allUsers[idx];
              final bool isUAdmin = u['role'] == 'ADMIN';
              final bool pReport = isUAdmin || (u['can_view_report'] ?? 0) == 1;
              final bool pAct = isUAdmin || (u['can_view_activity'] ?? 0) == 1;
              final bool pMenu = isUAdmin || (u['can_edit_menu'] ?? 0) == 1;
              final bool pDel = isUAdmin || (u['can_delete_order'] ?? 0) == 1;

              return Card(
                elevation: 1,
                shape: RoundedRectangleBorder(
                  borderRadius: BorderRadius.circular(8),
                  side: BorderSide(color: isUAdmin ? kOchaGreen : Colors.grey.shade300, width: isUAdmin ? 1.5 : 1),
                ),
                child: Padding(
                  padding: const EdgeInsets.all(12),
                  child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Row(
                        children: [
                          CircleAvatar(
                            backgroundColor: isUAdmin ? kOchaGreen : kOchaTakeawayBlue,
                            child: Icon(isUAdmin ? Icons.security : Icons.person, color: Colors.white),
                          ),
                          const SizedBox(width: 12),
                          Expanded(
                            child: Column(
                              crossAxisAlignment: CrossAxisAlignment.start,
                              children: [
                                Text('${{u['name']}}', style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                                Text('Vai trò: ${{isUAdmin ? "CHỦ QUÁN (ADMIN - Toàn quyền)" : "NHÂN VIÊN (USER)"}}  •  Mã PIN: ${{u['pin']}}', style: const TextStyle(color: Colors.black54, fontSize: 13)),
                              ],
                            ),
                          ),
                          OutlinedButton.icon(
                            icon: const Icon(Icons.login, size: 16),
                            label: const Text('Chuyển sang User này'),
                            onPressed: () {{
                              setState(() =""" + f"""> currentUser = Map{LT}String, dynamic{GT}.from(u));
                              ScaffoldMessenger.of(context).showSnackBar(
                                SnackBar(backgroundColor: kOchaGreen, content: Text('Đã chuyển ca sang: ${{u['name']}}')),
                              );
                            }},
                          ),
                          const SizedBox(width: 6),
                          IconButton(
                            icon: const Icon(Icons.edit, color: Colors.blue),
                            onPressed: () =""" + f"""> _showEditUserDialog(userObj: u),
                          ),
                          if (!isUAdmin)
                            IconButton(
                              icon: const Icon(Icons.delete_outline, color: Colors.red),
                              onPressed: () async {{
                                final db = await LocalDb.db;
                                await db.delete('users', where: 'id = ?', whereArgs: [u['id']]);
                                await _loadData();
                              }},
                            ),
                        ],
                      ),
                      const SizedBox(height: 8),
                      Wrap(
                        spacing: 10,
                        runSpacing: 6,
                        children: [
                          FilterChip(
                            label: const Text('📊 Xem Báo cáo Doanh thu'),
                            selected: pReport,
                            selectedColor: Colors.green.shade100,
                            onSelected: isUAdmin
                                ? null
                                : (val) async {{
                                    final db = await LocalDb.db;
                                    await db.update('users', {{'can_view_report': val ? 1 : 0}}, where: 'id = ?', whereArgs: [u['id']]);
                                    await _loadData();
                                  }},
                          ),
                          FilterChip(
                            label: const Text('🧾 Xem Lịch sử Biên lai'),
                            selected: pAct,
                            selectedColor: Colors.green.shade100,
                            onSelected: isUAdmin
                                ? null
                                : (val) async {{
                                    final db = await LocalDb.db;
                                    await db.update('users', {{'can_view_activity': val ? 1 : 0}}, where: 'id = ?', whereArgs: [u['id']]);
                                    await _loadData();
                                  }},
                          ),
                          FilterChip(
                            label: const Text('🍜 Sửa Thực đơn & Giá'),
                            selected: pMenu,
                            selectedColor: Colors.green.shade100,
                            onSelected: isUAdmin
                                ? null
                                : (val) async {{
                                    final db = await LocalDb.db;
                                    await db.update('users', {{'can_edit_menu': val ? 1 : 0}}, where: 'id = ?', whereArgs: [u['id']]);
                                    await _loadData();
                                  }},
                          ),
                          FilterChip(
                            label: const Text('🗑️ Xóa Hóa đơn đang mở'),
                            selected: pDel,
                            selectedColor: Colors.green.shade100,
                            onSelected: isUAdmin
                                ? null
                                : (val) async {{
                                    final db = await LocalDb.db;
                                    await db.update('users', {{'can_delete_order': val ? 1 : 0}}, where: 'id = ?', whereArgs: [u['id']]);
                                    await _loadData();
                                  }},
                          ),
                        ],
                      ),
                    ],
                  ),
                ),
              );
            }},
          ),
        ),
      ],
    );
  }}

  void _showEditUserDialog({{dynamic userObj}}) {{
    final bool isEdit = userObj != null;
    final nameCtrl = TextEditingController(text: isEdit ? userObj['name'].toString() : '');
    final pinCtrl = TextEditingController(text: isEdit ? userObj['pin'].toString() : '');
    String role = isEdit ? userObj['role'].toString() : 'STAFF';
    bool canRep = isEdit ? ((userObj['can_view_report'] ?? 0) == 1) : false;
    bool canAct = isEdit ? ((userObj['can_view_activity'] ?? 1) == 1) : true;
    bool canMenu = isEdit ? ((userObj['can_edit_menu'] ?? 0) == 1) : false;
    bool canDel = isEdit ? ((userObj['can_delete_order'] ?? 0) == 1) : false;

    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setUSt) =""" + f"""> AlertDialog(
          title: Text(isEdit ? 'Sửa thông tin & Phân quyền User' : 'Thêm Tài khoản Nhân viên mới'),
          content: SizedBox(
            width: 480,
            child: SingleChildScrollView(
              child: Column(
                mainAxisSize: MainAxisSize.min,
                children: [
                  TextField(
                    controller: nameCtrl,
                    decoration: const InputDecoration(labelText: 'Tên nhân viên (In trên hóa đơn)', border: OutlineInputBorder()),
                  ),
                  const SizedBox(height: 12),
                  TextField(
                    controller: pinCtrl,
                    keyboardType: TextInputType.number,
                    maxLength: 4,
                    decoration: const InputDecoration(labelText: 'Mã PIN đăng nhập (4 chữ số, VD: 3333)', border: OutlineInputBorder()),
                  ),
                  const SizedBox(height: 8),
                  DropdownButtonFormField{LT}String{GT}(
                    value: role,
                    decoration: const InputDecoration(labelText: 'Vai trò hệ thống', border: OutlineInputBorder()),
                    items: const [
                      DropdownMenuItem(value: 'ADMIN', child: Text('CHỦ QUÁN / ADMIN (Toàn quyền)')),
                      DropdownMenuItem(value: 'STAFF', child: Text('NHÂN VIÊN / USER (Phân quyền bên dưới)')),
                    ],
                    onChanged: (v) =""" + f"""> setUSt(() =""" + f"""> role = v ?? 'STAFF'),
                  ),
                  if (role != 'ADMIN') ...[
                    const SizedBox(height: 10),
                    CheckboxListTile(
                      title: const Text('Cho phép xem Báo cáo Doanh thu'),
                      value: canRep,
                      activeColor: kOchaGreen,
                      onChanged: (v) =""" + f"""> setUSt(() =""" + f"""> canRep = v ?? false),
                    ),
                    CheckboxListTile(
                      title: const Text('Cho phép xem Lịch sử Biên lai (Hoạt động)'),
                      value: canAct,
                      activeColor: kOchaGreen,
                      onChanged: (v) =""" + f"""> setUSt(() =""" + f"""> canAct = v ?? false),
                    ),
                    CheckboxListTile(
                      title: const Text('Cho phép Chỉnh sửa Thực đơn & Giá bán'),
                      value: canMenu,
                      activeColor: kOchaGreen,
                      onChanged: (v) =""" + f"""> setUSt(() =""" + f"""> canMenu = v ?? false),
                    ),
                    CheckboxListTile(
                      title: const Text('Cho phép Xóa Hóa đơn đang mở'),
                      value: canDel,
                      activeColor: kOchaGreen,
                      onChanged: (v) =""" + f"""> setUSt(() =""" + f"""> canDel = v ?? false),
                    ),
                  ],
                ],
              ),
            ),
          ),
          actions: [
            TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Hủy')),
            ElevatedButton(
              style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen),
              onPressed: () async {{
                if (nameCtrl.text.trim().isEmpty || pinCtrl.text.trim().length != 4) return;
                final db = await LocalDb.db;
                final row = {{
                  'name': nameCtrl.text.trim(),
                  'pin': pinCtrl.text.trim(),
                  'role': role,
                  'can_view_report': role == 'ADMIN' ? 1 : (canRep ? 1 : 0),
                  'can_view_activity': role == 'ADMIN' ? 1 : (canAct ? 1 : 0),
                  'can_edit_menu': role == 'ADMIN' ? 1 : (canMenu ? 1 : 0),
                  'can_delete_order': role == 'ADMIN' ? 1 : (canDel ? 1 : 0),
                }};
                if (isEdit) {{
                  await db.update('users', row, where: 'id = ?', whereArgs: [userObj['id']]);
                }} else {{
                  await db.insert('users', row);
                }}
                await _loadData();
                if (mounted) Navigator.pop(context);
              }},
              child: const Text('Lưu tài khoản'),
            ),
          ],
        ),
      ),
    );
  }}

  Widget _buildSoundSettingsPanel() {{
    final String fileNameOnly = customSoundPath.isEmpty ? 'Chưa chọn file nào' : p.basename(customSoundPath);
    return ListView(
      children: [
        const Text('Cấu hình Âm thanh khi In Hóa đơn & Mở Két tiền', style: TextStyle(fontSize: 19, fontWeight: FontWeight.bold)),
        const SizedBox(height: 14),
        Card(
          elevation: 0,
          shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(8), side: BorderSide(color: soundMode == 'OFF' ? kOchaOccupiedRed : Colors.grey.shade300, width: soundMode == 'OFF' ? 2 : 1)),
          child: RadioListTile{LT}String{GT}(
            value: 'OFF',
            groupValue: soundMode,
            activeColor: kOchaOccupiedRed,
            title: const Text('1. Tắt âm thanh (Im lặng hoàn toàn)', style: TextStyle(fontWeight: FontWeight.bold)),
            secondary: const Icon(Icons.volume_off, color: kOchaOccupiedRed),
            onChanged: (val) {{
              if (val == null) return;
              setState(() =""" + f"""> soundMode = val);
              CashDrawerBellSound.soundMode = val;
              _saveSetting('sound_mode', val);
            }},
          ),
        ),
        const SizedBox(height: 10),
        Card(
          elevation: 0,
          shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(8), side: BorderSide(color: soundMode == 'DEFAULT' ? kOchaGreen : Colors.grey.shade300, width: soundMode == 'DEFAULT' ? 2 : 1)),
          child: RadioListTile{LT}String{GT}(
            value: 'DEFAULT',
            groupValue: soundMode,
            activeColor: kOchaGreen,
            title: const Text('2. Dùng âm thanh Kịch-Ting chuẩn Ocha mặc định', style: TextStyle(fontWeight: FontWeight.bold)),
            secondary: OutlinedButton.icon(
              icon: const Icon(Icons.play_arrow, size: 18),
              label: const Text('Nghe thử'),
              onPressed: () =""" + f"""> CashDrawerBellSound.playSound(forceMode: 'DEFAULT'),
            ),
            onChanged: (val) {{
              if (val == null) return;
              setState(() =""" + f"""> soundMode = val);
              CashDrawerBellSound.soundMode = val;
              _saveSetting('sound_mode', val);
            }},
          ),
        ),
        const SizedBox(height: 10),
        Card(
          elevation: 0,
          shape: RoundedRectangleBorder(borderRadius: BorderRadius.circular(8), side: BorderSide(color: soundMode == 'CUSTOM' ? kOchaTakeawayBlue : Colors.grey.shade300, width: soundMode == 'CUSTOM' ? 2 : 1)),
          child: Padding(
            padding: const EdgeInsets.only(bottom: 12),
            child: Column(
              children: [
                RadioListTile{LT}String{GT}(
                  value: 'CUSTOM',
                  groupValue: soundMode,
                  activeColor: kOchaTakeawayBlue,
                  title: const Text('3. Tự tải file âm thanh riêng (.m4a, .mp3, .wav)', style: TextStyle(fontWeight: FontWeight.bold)),
                  subtitle: Text('File đang dùng: $fileNameOnly'),
                  onChanged: (val) {{
                    if (val == null) return;
                    setState(() =""" + f"""> soundMode = val);
                    CashDrawerBellSound.soundMode = val;
                    _saveSetting('sound_mode', val);
                  }},
                ),
                Padding(
                  padding: const EdgeInsets.symmetric(horizontal: 20),
                  child: Column(
                    crossAxisAlignment: CrossAxisAlignment.start,
                    children: [
                      Row(
                        children: [
                          ElevatedButton.icon(
                            style: ElevatedButton.styleFrom(backgroundColor: kOchaTakeawayBlue),
                            icon: const Icon(Icons.folder_open),
                            label: const Text('Chọn file âm thanh'),
                            onPressed: () async {{
                              final savedPath = await _pickAndSaveCustomAudioFile();
                              if (savedPath != null) {{
                                setState(() {{
                                  customSoundPath = savedPath;
                                  soundMode = 'CUSTOM';
                                }});
                                CashDrawerBellSound.customSoundPath = savedPath;
                                CashDrawerBellSound.soundMode = 'CUSTOM';
                                await _saveSetting('custom_sound_path', savedPath);
                                await _saveSetting('sound_mode', 'CUSTOM');
                                CashDrawerBellSound.playSound(forceMode: 'CUSTOM');
                              }}
                            }},
                          ),
                          const SizedBox(width: 12),
                          OutlinedButton.icon(
                            icon: const Icon(Icons.volume_up),
                            label: const Text('Phát thử file'),
                            onPressed: customSoundPath.isEmpty ? null : () =""" + f"""> CashDrawerBellSound.playSound(forceMode: 'CUSTOM'),
                          ),
                        ],
                      ),
                      const SizedBox(height: 12),
                      Text('Bắt đầu phát từ giây thứ: ${{customSoundOffset.toStringAsFixed(1)}}s'),
                      Slider(
                        value: customSoundOffset.clamp(0.0, 5.0),
                        min: 0.0,
                        max: 5.0,
                        divisions: 50,
                        activeColor: kOchaTakeawayBlue,
                        onChanged: (val) {{
                          setState(() =""" + f"""> customSoundOffset = val);
                          CashDrawerBellSound.customStartOffsetSec = val;
                          _saveSetting('custom_sound_offset', val.toStringAsFixed(2));
                        }},
                      ),
                    ],
                  ),
                ),
              ],
            ),
          ),
        ),
      ],
    );
  }}

  Widget _buildVersionPanel() {{
    return Column(
      crossAxisAlignment: CrossAxisAlignment.start,
      children: [
        Container(
          padding: const EdgeInsets.symmetric(horizontal: 14, vertical: 8),
          decoration: BoxDecoration(color: kOchaGreen, borderRadius: BorderRadius.circular(8)),
          child: const Text('OCHA LOCAL POS $APP_VERSION', style: TextStyle(color: Colors.white, fontWeight: FontWeight.bold, fontSize: 20)),
        ),
        const SizedBox(height: 16),
        Text(APP_BUILD_INFO, style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
        const SizedBox(height: 12),
        const Text('• Đăng nhập PIN 4 số & Phân quyền Admin/Nhân viên (Khóa/Mở quyền Xem Báo cáo)'),
        const SizedBox(height: 6),
        const Text('• Quản lý Thực đơn dạng Bảng tính thông minh + Tìm kiếm không dấu + Nhân bản món 1 chạm'),
        const SizedBox(height: 6),
        const Text('• Export / Import Thực đơn 2 chiều với Google Sheets & File CSV'),
        const SizedBox(height: 6),
        const Text('• Kiến trúc Non-blocking Print Queue (Bấm Lưu & Thanh toán phản hồi trong 0.05 giây)'),
      ],
    );
  }}

  Widget _buildTablesManagerSettingsPanel() {{
    return Column(
      crossAxisAlignment: CrossAxisAlignment.start,
      children: [
        Row(
          mainAxisAlignment: MainAxisAlignment.spaceBetween,
          children: [
            const Text('Quản lý Phân khu & Danh sách Bàn', style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
            Row(
              children: [
                ElevatedButton.icon(
                  style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen),
                  icon: const Icon(Icons.layers),
                  label: const Text('Tạo / Xóa Phân khu'),
                  onPressed: _showManageAreasModal,
                ),
                const SizedBox(width: 10),
                ElevatedButton.icon(
                  style: ElevatedButton.styleFrom(backgroundColor: kOchaTakeawayBlue),
                  icon: const Icon(Icons.add),
                  label: const Text('Tạo Bàn mới'),
                  onPressed: () =""" + f"""> _showCreateOrEditTableDialog(),
                ),
              ],
            ),
          ],
        ),
        const SizedBox(height: 12),
        Expanded(
          child: ListView.separated(
            itemCount: allDiningTables.length,
            separatorBuilder: (_, __) =""" + f"""> const Divider(height: 1),
            itemBuilder: (context, i) {{
              final t = allDiningTables[i];
              return ListTile(
                leading: CircleAvatar(
                  backgroundColor: kOchaGreen,
                  child: Text((t['table_code'] ?? '').toString(), style: const TextStyle(color: Colors.white, fontSize: 12, fontWeight: FontWeight.bold)),
                ),
                title: Text(t['table_name'].toString(), style: const TextStyle(fontWeight: FontWeight.bold)),
                subtitle: Text('Phân khu: ${{t['area']}}'),
                trailing: Row(
                  mainAxisSize: MainAxisSize.min,
                  children: [
                    IconButton(icon: const Icon(Icons.edit, color: Colors.blue), onPressed: () =""" + f"""> _showCreateOrEditTableDialog(tableObj: t)),
                    IconButton(
                      icon: const Icon(Icons.delete_outline, color: Colors.red),
                      onPressed: () async {{
                        final db = await LocalDb.db;
                        await db.delete('dining_tables', where: 'id = ?', whereArgs: [t['id']]);
                        await _loadData();
                      }},
                    ),
                  ],
                ),
              );
            }},
          ),
        ),
      ],
    );
  }}

  Widget _buildActivityScreen() {{
    final sel = selectedActivityOrder ?? (paidOrders.isNotEmpty ? paidOrders.first : null);
    final List selItems = sel != null ? jsonDecode(sel['items_json'].toString()) : [];

    return Scaffold(
      backgroundColor: Colors.white,
      body: Row(
        children: [
          Expanded(
            flex: 32,
            child: Column(
              children: [
                Container(
                  height: 54,
                  padding: const EdgeInsets.symmetric(horizontal: 8),
                  child: Row(
                    children: [
                      IconButton(icon: const Icon(Icons.arrow_back), onPressed: () =""" + f"""> setState(() =""" + f"""> fullScreenView = 'NONE')),
                      const Expanded(child: Center(child: Text('Hoạt động', style: TextStyle(fontSize: 18, fontWeight: FontWeight.bold)))),
                      const SizedBox(width: 40),
                    ],
                  ),
                ),
                const Divider(height: 1),
                Expanded(
                  child: paidOrders.isEmpty
                      ? const Center(child: Text('Chưa có giao dịch nào'))
                      : ListView.separated(
                          itemCount: paidOrders.length,
                          separatorBuilder: (_, __) =""" + f"""> const Divider(height: 1),
                          itemBuilder: (context, idx) {{
                            final o = paidOrders[idx];
                            final bool isSelected = sel != null && sel['id'] == o['id'];
                            return ListTile(
                              selected: isSelected,
                              selectedTileColor: Colors.blue.shade50,
                              leading: const Icon(Icons.monetization_on_outlined, color: Colors.blueGrey),
                              title: Text('${{fmt.format(o['total'])}}₫', style: const TextStyle(fontWeight: FontWeight.bold)),
                              subtitle: Text('${{o['table_name']}}  •  #${{o['receipt_no'] ?? o['id']}}'),
                              trailing: Text(
                                o['end_time'].toString().length {GT} 11 ? o['end_time'].toString().substring(11) : '',
                                style: const TextStyle(color: Colors.grey, fontSize: 12),
                              ),
                              onTap: () =""" + f"""> setState(() =""" + f"""> selectedActivityOrder = o),
                            );
                          }},
                        ),
                ),
              ],
            ),
          ),
          const VerticalDivider(width: 1),
          Expanded(
            flex: 68,
            child: sel == null
                ? const Center(child: Text('Chọn một biên lai bên trái để xem'))
                : Column(
                    children: [
                      Container(
                        height: 54,
                        padding: const EdgeInsets.symmetric(horizontal: 20),
                        child: Row(
                          mainAxisAlignment: MainAxisAlignment.spaceBetween,
                          children: [
                            const SizedBox(width: 40),
                            Text('${{fmt.format(sel['total'])}}₫ bán hàng', style: const TextStyle(fontSize: 18, fontWeight: FontWeight.bold)),
                            IconButton(
                              tooltip: 'In lại biên lai',
                              icon: const Icon(Icons.print, color: Color(0xFF4285F4)),
                              onPressed: () async {{
                                unawaited(OchaPrinter.printBill(
                                  storeName: storeName,
                                  storeAddress: storeAddress,
                                  storePhone: storePhone,
                                  receiptNo: (sel['receipt_no'] ?? '').toString(),
                                  orderType: (sel['order_type'] ?? 'Dùng tại quán').toString(),
                                  tableName: sel['table_name'].toString(),
                                  guestCount: (sel['guest_count'] ?? 1) as int,
                                  staffName: (sel['staff_name'] ?? currentStaffName).toString(),
                                  startTime: sel['start_time'].toString(),
                                  items: selItems,
                                  discount: (sel['discount'] ?? 0) as int,
                                  surcharge: (sel['surcharge'] ?? 0) as int,
                                  total: sel['total'] as int,
                                  paymentMethod: (sel['payment_method'] ?? 'Tiền mặt').toString(),
                                  note: (sel['note'] ?? '').toString(),
                                  printMode: 'PAID',
                                  openDrawer: false,
                                ));
                              }},
                            ),
                          ],
                        ),
                      ),
                      const Divider(height: 1),
                      Expanded(
                        child: ListView(
                          padding: const EdgeInsets.symmetric(horizontal: 40, vertical: 20),
                          children: [
                            const Center(child: Text('BIÊN LAI', style: TextStyle(color: Colors.black54, fontWeight: FontWeight.bold))),
                            const SizedBox(height: 16),
                            _buildReceiptDetailRow((sel['order_type'] ?? 'Dùng tại quán').toString(), sel['table_name'].toString()),
                            _buildReceiptDetailRow('Phục vụ bởi', (sel['staff_name'] ?? currentStaffName).toString()),
                            _buildReceiptDetailRow('Giờ vào', sel['start_time'].toString()),
                            _buildReceiptDetailRow('Giờ ra', sel['end_time'].toString()),
                            _buildReceiptDetailRow('Số biên lai', (sel['receipt_no'] ?? '').toString()),
                            _buildReceiptDetailRow('Mặt hàng', '${{fmt.format(sel['total'])}}₫'),
                            for (var it in selItems)
                              _buildReceiptDetailRow(
                                '  • ${{it['name']}} (x${{it['qty']}})',
                                '${{fmt.format((it['qty'] as int) * ((it['price'] as int) + ((it['modifier_price'] ?? 0) as int)))}}₫',
                              ),
                          ],
                        ),
                      ),
                    ],
                  ),
          ),
        ],
      ),
    );
  }}

  Widget _buildReceiptDetailRow(String left, String right) {{
    return Column(
      children: [
        Padding(
          padding: const EdgeInsets.symmetric(vertical: 12),
          child: Row(
            mainAxisAlignment: MainAxisAlignment.spaceBetween,
            children: [
              Text(left, style: const TextStyle(fontSize: 15, color: Colors.black87)),
              Text(right, style: const TextStyle(fontSize: 15, color: Colors.black54)),
            ],
          ),
        ),
        const Divider(height: 1),
      ],
    );
  }}

  Widget _buildReportScreen() {{
    final today = DateFormat('yyyy-MM-dd').format(DateTime.now());
    final todayOrders = paidOrders.where((o) =""" + f"""> o['end_time'].toString().startsWith(today)).toList();
    int totalToday = 0;
    int cashToday = 0;
    int transferToday = 0;
    for (var o in todayOrders) {{
      int t = o['total'] as int;
      totalToday += t;
      if (o['payment_method'] == 'Tiền mặt') {{
        cashToday += t;
      }} else {{
        transferToday += t;
      }}
    }}

    return Scaffold(
      appBar: AppBar(
        backgroundColor: Colors.white,
        foregroundColor: Colors.black87,
        elevation: 1,
        leading: IconButton(icon: const Icon(Icons.arrow_back), onPressed: () =""" + f"""> setState(() =""" + f"""> fullScreenView = 'NONE')),
        title: Text('Báo cáo Doanh thu ($today) • Người xem: ${{currentUser['name']}}'),
        actions: [
          ElevatedButton.icon(
            style: ElevatedButton.styleFrom(backgroundColor: kOchaTakeawayBlue),
            icon: const Icon(Icons.file_download),
            label: const Text('Xuất CSV Doanh thu'),
            onPressed: () async {{
              final dir = await getExternalStorageDirectory();
              final f = File('${{dir!.path}}/DoanhThu_$today.csv');
              final sb = StringBuffer('\\uFEFFMã Biên Lai,Giờ Ra,Bàn,Nhân Viên,Phương Thức,Tổng Tiền\\n');
              for (var o in todayOrders) {{
                sb.writeln('"${{o['receipt_no']}}","${{o['end_time']}}","${{o['table_name']}}","${{o['staff_name']}}","${{o['payment_method']}}",${{o['total']}}');
              }}
              await f.writeAsString(sb.toString());
              if (mounted) {{
                ScaffoldMessenger.of(context).showSnackBar(SnackBar(content: Text('Đã lưu CSV tại: ${{f.path}}')));
              }}
            }},
          ),
          const SizedBox(width: 16),
        ],
      ),
      body: Center(
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            Text('Tổng số đơn đã thanh toán hôm nay: ${{todayOrders.length}} đơn', style: const TextStyle(fontSize: 22)),
            const SizedBox(height: 10),
            Text('Tiền mặt: ${{fmt.format(cashToday)}}₫   |   Chuyển khoản / App: ${{fmt.format(transferToday)}}₫', style: const TextStyle(fontSize: 18, color: Colors.black54)),
            const SizedBox(height: 16),
            Text('TỔNG DOANH THU: ${{fmt.format(totalToday)}}₫', style: const TextStyle(fontSize: 34, fontWeight: FontWeight.bold, color: kOchaGreen)),
          ],
        ),
      ),
    );
  }}

  // ==================== CÁC HỘP THOẠI PHỤ TRỢ ====================
  void _showManageAreasModal() {{
    final areaCtrl = TextEditingController();
    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setAreaSt) =""" + f"""> AlertDialog(
          title: const Text('Quản lý Phân khu bàn (Tạo / Xóa khu vực)'),
          content: SizedBox(
            width: 520,
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                Row(
                  children: [
                    Expanded(
                      child: TextField(
                        controller: areaCtrl,
                        decoration: const InputDecoration(labelText: 'Tên phân khu mới (VD: Tầng trệt, Lầu 1...)', border: OutlineInputBorder(), isDense: true),
                      ),
                    ),
                    const SizedBox(width: 10),
                    ElevatedButton.icon(
                      style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen, padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 14)),
                      icon: const Icon(Icons.add),
                      label: const Text('Tạo khu vực'),
                      onPressed: () async {{
                        final newArea = areaCtrl.text.trim();
                        if (newArea.isEmpty) return;
                        final db = await LocalDb.db;
                        await db.insert('table_areas', {{'area_name': newArea}}, conflictAlgorithm: ConflictAlgorithm.ignore);
                        areaCtrl.clear();
                        await _loadData();
                        setAreaSt(() {{}});
                      }},
                    ),
                  ],
                ),
                const SizedBox(height: 14),
                SizedBox(
                  height: 220,
                  child: ListView.separated(
                    itemCount: allAreaNames.length,
                    separatorBuilder: (_, __) =""" + f"""> const Divider(height: 1),
                    itemBuilder: (context, i) {{
                      final aName = allAreaNames[i];
                      final int countTbl = allDiningTables.where((t) =""" + f"""> t['area'] == aName).length;
                      return ListTile(
                        leading: const Icon(Icons.layers, color: kOchaGreen),
                        title: Text(aName, style: const TextStyle(fontWeight: FontWeight.bold)),
                        subtitle: Text('Đang có: $countTbl bàn'),
                        trailing: IconButton(
                          icon: const Icon(Icons.delete_outline, color: Colors.red),
                          onPressed: allAreaNames.length {LT}= 1
                              ? null
                              : () async {{
                                  final db = await LocalDb.db;
                                  await db.delete('table_areas', where: 'area_name = ?', whereArgs: [aName]);
                                  await db.delete('dining_tables', where: 'area = ?', whereArgs: [aName]);
                                  await _loadData();
                                  setAreaSt(() {{}});
                                }},
                        ),
                      );
                    }},
                  ),
                ),
              ],
            ),
          ),
          actions: [TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Đóng'))],
        ),
      ),
    );
  }}

  void _showCreateOrEditTableDialog({{dynamic tableObj, String? defaultArea}}) {{
    final bool isEdit = tableObj != null;
    String chosenArea = isEdit ? tableObj['area'].toString() : (defaultArea ?? (allAreaNames.isNotEmpty ? allAreaNames.first : 'Bàn'));
    if (!allAreaNames.contains(chosenArea) && allAreaNames.isNotEmpty) chosenArea = allAreaNames.first;

    final int nextNum = allDiningTables.where((t) =""" + f"""> t['area'] == chosenArea).length + 1;
    final codeCtrl = TextEditingController(text: isEdit ? (tableObj['table_code'] ?? '').toString() : '$nextNum');
    final nameCtrl = TextEditingController(text: isEdit ? tableObj['table_name'].toString() : '$chosenArea - $nextNum');

    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setTblSt) =""" + f"""> AlertDialog(
          title: Text(isEdit ? 'Chỉnh sửa thông tin bàn' : 'Tạo bàn mới theo phân khu'),
          content: SizedBox(
            width: 460,
            child: Column(
              mainAxisSize: MainAxisSize.min,
              children: [
                DropdownButtonFormField{LT}String{GT}(
                  value: chosenArea,
                  decoration: const InputDecoration(labelText: 'Thuộc phân khu', border: OutlineInputBorder()),
                  items: allAreaNames.map((a) =""" + f"""> DropdownMenuItem(value: a, child: Text(a))).toList(),
                  onChanged: (val) {{
                    if (val == null) return;
                    final int cnt = allDiningTables.where((t) =""" + f"""> t['area'] == val).length + 1;
                    setTblSt(() {{
                      chosenArea = val;
                      if (!isEdit) {{
                        codeCtrl.text = '$cnt';
                        nameCtrl.text = '$val - $cnt';
                      }}
                    }});
                  }},
                ),
                const SizedBox(height: 12),
                TextField(
                  controller: codeCtrl,
                  decoration: const InputDecoration(labelText: 'Số hiệu bàn (Giữa mặt bàn, VD: 1, 8t)', border: OutlineInputBorder()),
                  onChanged: (v) {{
                    if (!isEdit) nameCtrl.text = '$chosenArea - ${{v.trim()}}';
                  }},
                ),
                const SizedBox(height: 12),
                TextField(
                  controller: nameCtrl,
                  decoration: const InputDecoration(labelText: 'Tên bàn in trên hóa đơn', border: OutlineInputBorder()),
                ),
              ],
            ),
          ),
          actions: [
            if (isEdit)
              TextButton(
                onPressed: () async {{
                  final db = await LocalDb.db;
                  await db.delete('dining_tables', where: 'id = ?', whereArgs: [tableObj['id']]);
                  await _loadData();
                  if (mounted) Navigator.pop(context);
                }},
                child: const Text('Xóa bàn', style: TextStyle(color: Colors.red)),
              ),
            TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Hủy')),
            ElevatedButton(
              style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen),
              onPressed: () async {{
                if (nameCtrl.text.trim().isEmpty) return;
                final db = await LocalDb.db;
                final row = {{
                  'area': chosenArea,
                  'table_code': codeCtrl.text.trim().isEmpty ? '1' : codeCtrl.text.trim(),
                  'table_name': nameCtrl.text.trim(),
                }};
                if (isEdit) {{
                  await db.update('dining_tables', row, where: 'id = ?', whereArgs: [tableObj['id']]);
                }} else {{
                  await db.insert('dining_tables', row);
                }}
                await _loadData();
                if (mounted) Navigator.pop(context);
              }},
              child: Text(isEdit ? 'Lưu' : 'Tạo bàn'),
            ),
          ],
        ),
      ),
    );
  }}

  Future{LT}String?{GT} _showSelectTableVisualModal({{required bool forSavingNow}}) async {{
    String dialogArea = allAreaNames.contains(activeArea) ? activeArea : (allAreaNames.isNotEmpty ? allAreaNames.first : 'Bàn');

    return await showDialog{LT}String{GT}(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setDlgSt) =""" + f"""> Dialog(
          insetPadding: const EdgeInsets.symmetric(horizontal: 40, vertical: 24),
          child: Container(
            width: 940,
            height: 560,
            padding: const EdgeInsets.all(16),
            child: Column(
              children: [
                Row(
                  children: [
                    Text(
                      forSavingNow ? 'Chọn Bàn hoặc Mang về để Lưu & In Bếp' : 'Sơ đồ Chọn Bàn theo Khu vực',
                      style: const TextStyle(fontSize: 18, fontWeight: FontWeight.bold),
                    ),
                    const Spacer(),
                    ElevatedButton.icon(
                      style: ElevatedButton.styleFrom(backgroundColor: kOchaTakeawayBlue, padding: const EdgeInsets.symmetric(horizontal: 18, vertical: 10)),
                      icon: const Icon(Icons.shopping_bag_outlined, size: 18),
                      label: const Text('🛍 ĐƠN MANG VỀ (Tự đánh Số 001, 002...)', style: TextStyle(fontWeight: FontWeight.bold)),
                      onPressed: () {{
                        final tk = _generateNextTakeawayInfo();
                        setState(() {{
                          activeOrderType = 'Mang về';
                          activeArea = 'Mang về';
                          activeTableName = tk['name']!;
                          activeTakeawayCode = tk['code']!;
                        }});
                        Navigator.pop(context, tk['name']);
                      }},
                    ),
                    const SizedBox(width: 12),
                    IconButton(icon: const Icon(Icons.close), onPressed: () =""" + f"""> Navigator.pop(context, null)),
                  ],
                ),
                const SizedBox(height: 8),
                SizedBox(
                  height: 40,
                  child: ListView.separated(
                    scrollDirection: Axis.horizontal,
                    itemCount: allAreaNames.length,
                    separatorBuilder: (_, __) =""" + f"""> const SizedBox(width: 8),
                    itemBuilder: (context, i) {{
                      final aName = allAreaNames[i];
                      final bool sel = dialogArea == aName;
                      return ChoiceChip(
                        label: Text(aName, style: TextStyle(fontWeight: FontWeight.bold, color: sel ? Colors.white : Colors.black87)),
                        selected: sel,
                        selectedColor: kOchaGreen,
                        onSelected: (_) =""" + f"""> setDlgSt(() =""" + f"""> dialogArea = aName),
                      );
                    }},
                  ),
                ),
                const Divider(),
                Expanded(child: _buildOchaVisualMapForArea(dialogArea, isInsideDialog: true)),
              ],
            ),
          ),
        ),
      ),
    );
  }}

  void _showOrderMoreActionsPopup() {{
    final nowDisplay = DateFormat('dd-MM-yyyy HH:mm').format(DateTime.now());
    showDialog(
      context: context,
      builder: (context) =""" + f"""> Dialog(
        alignment: Alignment.centerRight,
        insetPadding: const EdgeInsets.only(right: 20, top: 45, bottom: 75),
        child: Container(
          width: 280,
          padding: const EdgeInsets.symmetric(vertical: 10),
          child: Column(
            mainAxisSize: MainAxisSize.min,
            children: [
              ListTile(
                leading: const Icon(Icons.send_outlined),
                title: const Text('Bếp'),
                onTap: () async {{
                  Navigator.pop(context);
                  for (var it in cart) it['printed_qty'] = 0;
                  await _handlePressSaveButton();
                }},
              ),
              ListTile(
                leading: const Icon(Icons.swap_horiz),
                title: const Text('Di chuyển bàn'),
                onTap: () {{
                  Navigator.pop(context);
                  _showSelectTableVisualModal(forSavingNow: false);
                }},
              ),
              ListTile(
                leading: const Icon(Icons.call_split),
                title: const Text('Tách hóa đơn'),
                onTap: () {{
                  Navigator.pop(context);
                  _showSplitOrderModal();
                }},
              ),
              ListTile(
                leading: const Icon(Icons.merge_type),
                title: const Text('Gộp hóa đơn'),
                onTap: () {{
                  Navigator.pop(context);
                  _showMergeOrderModal();
                }},
              ),
              ListTile(
                leading: const Icon(Icons.note_add_outlined),
                title: const Text('Thêm đơn'),
                onTap: () {{
                  Navigator.pop(context);
                  _startNewOrder();
                }},
              ),
              ListTile(
                leading: const Icon(Icons.edit_outlined),
                title: const Text('Ghi chú hóa đơn'),
                onTap: () {{
                  Navigator.pop(context);
                  _showOrderNoteDialog();
                }},
              ),
              ListTile(
                leading: const Icon(Icons.delete_forever_outlined, color: Colors.red),
                title: const Text('Xóa hóa đơn', style: TextStyle(color: Colors.red)),
                onTap: () async {{
                  Navigator.pop(context);
                  if (!await _verifyPermissionOrAskAdminPin(canDeleteOrder, 'Xóa hóa đơn')) return;
                  if (activeOrderId != null) {{
                    final db = await LocalDb.db;
                    await db.delete('orders', where: 'id = ?', whereArgs: [activeOrderId]);
                    await _loadData();
                  }}
                  setState(() {{
                    activeOrderId = null;
                    cart.clear();
                    currentBottomTab = 'Hóa đơn';
                  }});
                }},
              ),
              const Divider(),
              Padding(
                padding: const EdgeInsets.symmetric(horizontal: 16, vertical: 4),
                child: Row(
                  mainAxisAlignment: MainAxisAlignment.spaceBetween,
                  children: [
                    const Text('Bắt đầu', style: TextStyle(color: Colors.grey, fontSize: 13)),
                    Text(nowDisplay, style: const TextStyle(color: Colors.grey, fontSize: 13)),
                  ],
                ),
              ),
            ],
          ),
        ),
      ),
    );
  }}

  void _showSplitOrderModal() {{
    if (cart.isEmpty) return;
    final Map{LT}int, int{GT} splitQtyMap = {{}};
    for (int i = 0; i {LT} cart.length; i++) splitQtyMap[i] = 0;
    String targetSplitTable = '${{activeTableName}} (Tách)';

    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setSplitSt) {{
          int splitTotal = 0;
          for (int i = 0; i {LT} cart.length; i++) {{
            final int sQty = splitQtyMap[i] ?? 0;
            final int unit = (cart[i]['price'] as int) + ((cart[i]['modifier_price'] ?? 0) as int);
            splitTotal += sQty * unit;
          }}

          return AlertDialog(
            title: const Text('Tách hóa đơn'),
            content: SizedBox(
              width: 580,
              height: 360,
              child: Column(
                children: [
                  Expanded(
                    child: ListView.separated(
                      itemCount: cart.length,
                      separatorBuilder: (_, __) =""" + f"""> const Divider(),
                      itemBuilder: (context, idx) {{
                        final it = cart[idx];
                        final int maxQty = it['qty'] as int;
                        final int currSplit = splitQtyMap[idx] ?? 0;
                        return ListTile(
                          title: Text('${{it['name']}} (Đang có: $maxQty)'),
                          trailing: Row(
                            mainAxisSize: MainAxisSize.min,
                            children: [
                              IconButton(
                                icon: const Icon(Icons.remove_circle_outline, color: Colors.red),
                                onPressed: currSplit == 0 ? null : () =""" + f"""> setSplitSt(() =""" + f"""> splitQtyMap[idx] = currSplit - 1),
                              ),
                              Text('$currSplit / $maxQty', style: const TextStyle(fontSize: 16, fontWeight: FontWeight.bold)),
                              IconButton(
                                icon: const Icon(Icons.add_circle_outline, color: Colors.green),
                                onPressed: currSplit {GT}= maxQty ? null : () =""" + f"""> setSplitSt(() =""" + f"""> splitQtyMap[idx] = currSplit + 1),
                              ),
                            ],
                          ),
                        );
                      }},
                    ),
                  ),
                  const Divider(),
                  Text('Tổng phần tách: ${{fmt.format(splitTotal)}}₫', style: const TextStyle(fontSize: 17, fontWeight: FontWeight.bold, color: Colors.deepOrange)),
                ],
              ),
            ),
            actions: [
              TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Hủy')),
              ElevatedButton(
                onPressed: splitTotal == 0
                    ? null
                    : () async {{
                        final List newSplitItems = [];
                        final List remainingItems = [];
                        for (int i = 0; i {LT} cart.length; i++) {{
                          final it = Map{LT}String, dynamic{GT}.from(cart[i]);
                          final int sQty = splitQtyMap[i] ?? 0;
                          final int remQty = (it['qty'] as int) - sQty;
                          if (sQty {GT} 0) {{
                            final spItem = Map{LT}String, dynamic{GT}.from(it);
                            spItem['qty'] = sQty;
                            spItem['printed_qty'] = sQty;
                            newSplitItems.add(spItem);
                          }}
                          if (remQty {GT} 0) {{
                            it['qty'] = remQty;
                            it['printed_qty'] = remQty;
                            remainingItems.add(it);
                          }}
                        }}

                        setState(() =""" + f"""> cart = remainingItems);
                        await _saveOpenOrderAndPrintKitchen();

                        final db = await LocalDb.db;
                        final now = DateFormat('yyyy-MM-dd HH:mm:ss').format(DateTime.now());
                        await db.insert('orders', {{
                          'receipt_no': '${{activeReceiptNo}}T',
                          'takeaway_code': '',
                          'order_type': activeOrderType,
                          'area': activeArea,
                          'table_name': targetSplitTable,
                          'guest_count': 1,
                          'staff_name': currentStaffName,
                          'note': '',
                          'items_json': jsonEncode(newSplitItems),
                          'discount': 0,
                          'surcharge': 0,
                          'total': splitTotal,
                          'payment_method': 'Tiền mặt',
                          'status': 'OPEN',
                          'prebill_printed': 0,
                          'start_time': now,
                          'end_time': '',
                          'synced': 0,
                        }});
                        await _loadData();
                        if (mounted) Navigator.pop(context);
                      }},
                child: const Text('Tách hóa đơn'),
              ),
            ],
          );
        }},
      ),
    );
  }}

  void _showPaymentCheckoutModal({{dynamic directOrder}}) {{
    final int billTotal = directOrder != null ? (directOrder['total'] as int) : cartTotal;
    final String tblLabel = directOrder != null ? directOrder['table_name'].toString() : activeTableName;
    int customerGiven = billTotal;
    String method = 'Tiền mặt';
    final givenCtrl = TextEditingController(text: billTotal.toString());
    final suggestions = [billTotal, 50000, 100000, 200000, 500000];

    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setPaySt) {{
          final int change = (customerGiven - billTotal).isNegative ? 0 : (customerGiven - billTotal);
          return AlertDialog(
            title: Row(
              mainAxisAlignment: MainAxisAlignment.spaceBetween,
              children: [
                Text('Thanh toán: $tblLabel'),
                Text('${{fmt.format(billTotal)}}₫', style: const TextStyle(color: kOchaGreen, fontWeight: FontWeight.bold, fontSize: 22)),
              ],
            ),
            content: SizedBox(
              width: 540,
              child: Column(
                mainAxisSize: MainAxisSize.min,
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(
                    children: ['Tiền mặt', 'Chuyển khoản', 'ShopeeFood', 'GrabFood'].map((m) {{
                      final bool active = method == m;
                      return Expanded(
                        child: Padding(
                          padding: const EdgeInsets.only(right: 6),
                          child: ElevatedButton(
                            style: ElevatedButton.styleFrom(
                              backgroundColor: active ? kOchaGreen : Colors.grey.shade300,
                              foregroundColor: active ? Colors.white : Colors.black87,
                            ),
                            onPressed: () =""" + f"""> setPaySt(() =""" + f"""> method = m),
                            child: Text(m, style: const TextStyle(fontSize: 13)),
                          ),
                        ),
                      );
                    }}).toList(),
                  ),
                  if (method == 'Tiền mặt') ...[
                    const SizedBox(height: 14),
                    Wrap(
                      spacing: 8,
                      children: suggestions.toSet().map((amt) {{
                        return OutlinedButton(
                          onPressed: () {{
                            setPaySt(() {{
                              customerGiven = amt;
                              givenCtrl.text = amt.toString();
                            }});
                          }},
                          child: Text(amt == billTotal ? 'Đúng số tiền' : '${{fmt.format(amt)}}₫'),
                        );
                      }}).toList(),
                    ),
                    const SizedBox(height: 12),
                    Row(
                      children: [
                        Expanded(
                          child: TextField(
                            controller: givenCtrl,
                            keyboardType: TextInputType.number,
                            decoration: const InputDecoration(labelText: 'Khách đưa (VNĐ)', border: OutlineInputBorder()),
                            onChanged: (v) =""" + f"""> setPaySt(() =""" + f"""> customerGiven = int.tryParse(v.trim()) ?? 0),
                          ),
                        ),
                        const SizedBox(width: 16),
                        Column(
                          crossAxisAlignment: CrossAxisAlignment.start,
                          children: [
                            const Text('Tiền thừa trả khách:', style: TextStyle(color: Colors.black54)),
                            Text('${{fmt.format(change)}}₫', style: const TextStyle(fontSize: 20, fontWeight: FontWeight.bold, color: kOchaGreen)),
                          ],
                        ),
                      ],
                    ),
                  ],
                ],
              ),
            ),
            actions: [
              TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Hủy')),
              ElevatedButton.icon(
                style: ElevatedButton.styleFrom(backgroundColor: kOchaGreen, padding: const EdgeInsets.symmetric(horizontal: 20, vertical: 12)),
                icon: const Icon(Icons.print),
                label: const Text('HOÀN TẤT & IN BIÊN LAI'),
                onPressed: () {{
                  Navigator.pop(context);
                  _payOrderNow(directOpenOrder: directOrder, method: method);
                }},
              ),
            ],
          );
        }},
      ),
    );
  }}

  void _showEditCartItemModifiersDialog(int idx) {{
    final item = cart[idx];
    final noteCtrl = TextEditingController(text: (item['note'] ?? '').toString());
    int qty = item['qty'] as int;
    final List{LT}String{GT} selectedMods = (item['modifiers'] ?? '').toString().split(', ').where((s) =""" + f"""> s.isNotEmpty).toList();

    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setSt) =""" + f"""> AlertDialog(
          title: Text('Tùy chọn món: ${{item['name']}}'),
          content: SizedBox(
            width: 520,
            child: SingleChildScrollView(
              child: Column(
                mainAxisSize: MainAxisSize.min,
                crossAxisAlignment: CrossAxisAlignment.start,
                children: [
                  Row(
                    mainAxisAlignment: MainAxisAlignment.center,
                    children: [
                      IconButton(icon: const Icon(Icons.remove_circle, color: Colors.red, size: 32), onPressed: () =""" + f"""> setSt(() =""" + f"""> qty = qty {GT} 1 ? qty - 1 : 1)),
                      Padding(padding: const EdgeInsets.symmetric(horizontal: 20), child: Text('$qty', style: const TextStyle(fontSize: 24, fontWeight: FontWeight.bold))),
                      IconButton(icon: const Icon(Icons.add_circle, color: Colors.green, size: 32), onPressed: () =""" + f"""> setSt(() =""" + f"""> qty += 1)),
                    ],
                  ),
                  const SizedBox(height: 10),
                  const Text('Tùy chọn thêm / Modifiers:', style: TextStyle(fontWeight: FontWeight.bold)),
                  const SizedBox(height: 8),
                  Wrap(
                    spacing: 8,
                    runSpacing: 8,
                    children: ochaModifiers.map((mod) {{
                      final String mName = mod['name'].toString();
                      final int mPrice = mod['price'] as int;
                      final String label = mPrice == 0 ? mName : '$mName (+${{fmt.format(mPrice)}}₫)';
                      final bool isChosen = selectedMods.contains(mName);
                      return FilterChip(
                        label: Text(label),
                        selected: isChosen,
                        selectedColor: Colors.orange.shade100,
                        onSelected: (v) {{
                          setSt(() {{
                            if (v) selectedMods.add(mName); else selectedMods.remove(mName);
                          }});
                        }},
                      );
                    }}).toList(),
                  ),
                  const SizedBox(height: 14),
                  TextField(
                    controller: noteCtrl,
                    decoration: const InputDecoration(labelText: 'Ghi chú riêng cho bếp', border: OutlineInputBorder()),
                  ),
                ],
              ),
            ),
          ),
          actions: [
            TextButton(
              onPressed: () {{
                _saveUndoSnapshot();
                setState(() =""" + f"""> cart.removeAt(idx));
                Navigator.pop(context);
              }},
              child: const Text('Xóa món', style: TextStyle(color: Colors.red)),
            ),
            ElevatedButton(
              onPressed: () {{
                int extraSum = 0;
                for (var m in ochaModifiers) {{
                  if (selectedMods.contains(m['name'])) extraSum += (m['price'] as int);
                }}
                setState(() {{
                  cart[idx]['qty'] = qty;
                  cart[idx]['modifiers'] = selectedMods.join(', ');
                  cart[idx]['modifier_price'] = extraSum;
                  cart[idx]['note'] = noteCtrl.text.trim();
                }});
                Navigator.pop(context);
              }},
              child: const Text('Lưu tùy chọn'),
            ),
          ],
        ),
      ),
    );
  }}

  void _showGuestCountModal() {{
    showDialog(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: const Text('Số lượng khách'),
        content: Wrap(
          spacing: 10,
          runSpacing: 10,
          children: [1, 2, 3, 4, 5, 6, 8, 10, 12].map((n) {{
            return ElevatedButton(
              onPressed: () {{
                setState(() =""" + f"""> activeGuestCount = n);
                Navigator.pop(context);
              }},
              child: Text('$n khách'),
            );
          }}).toList(),
        ),
      ),
    );
  }}

  void _showEditDishDialog({{dynamic product, String? defaultTab}}) {{
    final bool isEdit = product != null;
    final List{LT}String{GT} catNames = allCategories.isEmpty ? ['CHUẨN'] : allCategories.map((c) =""" + f"""> c['name'].toString()).toList();
    String selectedTab = isEdit ? product['bottom_tab'].toString() : (defaultTab ?? catNames.first);
    if (!catNames.contains(selectedTab)) selectedTab = catNames.first;

    final codeCtrl = TextEditingController(text: isEdit ? product['short_code'].toString() : 'BT');
    final nameCtrl = TextEditingController(text: isEdit ? product['name'].toString() : '');
    final priceCtrl = TextEditingController(text: isEdit ? product['price'].toString() : '45000');
    final imgCtrl = TextEditingController(text: isEdit ? (product['image_path'] ?? '').toString() : '');
    bool inStock = isEdit ? ((product['in_stock'] ?? 1) == 1) : true;

    showDialog(
      context: context,
      builder: (context) =""" + f"""> StatefulBuilder(
        builder: (context, setModal) =""" + f"""> AlertDialog(
          title: Text(isEdit ? 'Sửa món ăn / Hình ảnh' : 'Thêm món ăn mới'),
          content: SizedBox(
            width: 500,
            child: SingleChildScrollView(
              child: Column(
                mainAxisSize: MainAxisSize.min,
                children: [
                  Row(
                    children: [
                      SizedBox(width: 90, height: 75, child: _buildOchaTileTopGraphic(imgCtrl.text.trim(), codeCtrl.text.trim())),
                      const SizedBox(width: 14),
                      Expanded(
                        child: Column(
                          crossAxisAlignment: CrossAxisAlignment.start,
                          children: [
                            ElevatedButton.icon(
                              icon: const Icon(Icons.image),
                              label: const Text('Chọn hình từ máy Sunmi / USB'),
                              onPressed: () async {{
                                final pth = await _pickAndSaveDishImage();
                                if (pth != null) setModal(() =""" + f"""> imgCtrl.text = pth);
                              }},
                            ),
                            if (imgCtrl.text.isNotEmpty)
                              TextButton(
                                onPressed: () =""" + f"""> setModal(() =""" + f"""> imgCtrl.text = ''),
                                child: const Text('Xóa ảnh (Dùng ô chữ viết tắt)', style: TextStyle(color: Colors.red)),
                              ),
                          ],
                        ),
                      ),
                    ],
                  ),
                  SwitchListTile(
                    title: const Text('Còn hàng phục vụ', style: TextStyle(fontWeight: FontWeight.bold)),
                    value: inStock,
                    activeColor: kOchaGreen,
                    onChanged: (v) =""" + f"""> setModal(() =""" + f"""> inStock = v),
                  ),
                  const SizedBox(height: 8),
                  DropdownButtonFormField{LT}String{GT}(
                    value: selectedTab,
                    decoration: const InputDecoration(labelText: 'Thuộc Danh mục (Tab dưới đáy)', border: OutlineInputBorder()),
                    items: catNames.map((t) =""" + f"""> DropdownMenuItem(value: t, child: Text(t))).toList(),
                    onChanged: (v) =""" + f"""> setModal(() =""" + f"""> selectedTab = v ?? catNames.first),
                  ),
                  const SizedBox(height: 10),
                  TextField(
                    controller: codeCtrl,
                    decoration: const InputDecoration(labelText: 'Chữ viết tắt giữa ô (VD: BT, ĐB, TT, TC, TÔ)', border: OutlineInputBorder()),
                    onChanged: (_) =""" + f"""> setModal(() {{}}),
                  ),
                  const SizedBox(height: 10),
                  TextField(
                    controller: nameCtrl,
                    decoration: const InputDecoration(labelText: 'Tên món hiển thị', border: OutlineInputBorder()),
                  ),
                  const SizedBox(height: 10),
                  TextField(
                    controller: priceCtrl,
                    keyboardType: TextInputType.number,
                    decoration: const InputDecoration(labelText: 'Giá bán (VNĐ)', border: OutlineInputBorder()),
                  ),
                ],
              ),
            ),
          ),
          actions: [
            TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Hủy')),
            ElevatedButton(
              onPressed: () async {{
                if (nameCtrl.text.trim().isEmpty) return;
                final db = await LocalDb.db;
                final row = {{
                  'bottom_tab': selectedTab,
                  'short_code': codeCtrl.text.trim().toUpperCase(),
                  'name': nameCtrl.text.trim(),
                  'price': int.tryParse(priceCtrl.text.trim()) ?? 0,
                  'image_path': imgCtrl.text.trim(),
                  'in_stock': inStock ? 1 : 0,
                }};
                if (isEdit) {{
                  await db.update('products', row, where: 'id = ?', whereArgs: [product['id']]);
                }} else {{
                  await db.insert('products', row);
                }}
                await _loadData();
                if (mounted) Navigator.pop(context);
              }},
              child: const Text('Lưu món'),
            ),
          ],
        ),
      ),
    );
  }}

  void _showCustomAmountDialog() {{
    final nameCtrl = TextEditingController(text: 'Món tùy chỉnh');
    final priceCtrl = TextEditingController();
    showDialog(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: const Text('Thêm số tiền tuỳ chỉnh'),
        content: Column(
          mainAxisSize: MainAxisSize.min,
          children: [
            TextField(controller: nameCtrl, decoration: const InputDecoration(labelText: 'Tên khoản thu')),
            TextField(controller: priceCtrl, keyboardType: TextInputType.number, autofocus: true, decoration: const InputDecoration(labelText: 'Số tiền (VNĐ)')),
          ],
        ),
        actions: [
          TextButton(onPressed: () =""" + f"""> Navigator.pop(context), child: const Text('Hủy')),
          ElevatedButton(
            onPressed: () {{
              final amt = int.tryParse(priceCtrl.text.trim()) ?? 0;
              if (amt {GT} 0) _addProductToCart({{'id': 9999, 'name': nameCtrl.text, 'price': amt, 'in_stock': 1}});
              Navigator.pop(context);
            }},
            child: const Text('Thêm vào hóa đơn'),
          ),
        ],
      ),
    );
  }}

  void _showDiscountDialog() {{
    final ctrl = TextEditingController(text: activeDiscount.toString());
    showDialog(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: const Text('Giảm giá đơn hàng'),
        content: Column(
          mainAxisSize: MainAxisSize.min,
          children: [
            Wrap(
              spacing: 8,
              children: [5, 10, 15, 20].map((pct) {{
                return OutlinedButton(
                  onPressed: () {{
                    setState(() =""" + f"""> activeDiscount = ((cartSubtotal * pct) / 100).round());
                    Navigator.pop(context);
                  }},
                  child: Text('Giảm $pct%'),
                );
              }}).toList(),
            ),
            const SizedBox(height: 10),
            TextField(controller: ctrl, keyboardType: TextInputType.number, decoration: const InputDecoration(labelText: 'Số tiền giảm trực tiếp (VNĐ)')),
          ],
        ),
        actions: [
          ElevatedButton(
            onPressed: () {{
              setState(() =""" + f"""> activeDiscount = int.tryParse(ctrl.text.trim()) ?? 0);
              Navigator.pop(context);
            }},
            child: const Text('Áp dụng'),
          ),
        ],
      ),
    );
  }}

  void _showSurchargeDialog() {{
    final ctrl = TextEditingController(text: activeSurcharge.toString());
    showDialog(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: const Text('Phụ phí'),
        content: TextField(controller: ctrl, keyboardType: TextInputType.number, decoration: const InputDecoration(labelText: 'Số tiền phụ phí (VNĐ)')),
        actions: [
          ElevatedButton(
            onPressed: () {{
              setState(() =""" + f"""> activeSurcharge = int.tryParse(ctrl.text.trim()) ?? 0);
              Navigator.pop(context);
            }},
            child: const Text('Áp dụng'),
          ),
        ],
      ),
    );
  }}

  void _showOrderNoteDialog() {{
    final ctrl = TextEditingController(text: activeOrderNote);
    showDialog(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: const Text('Ghi chú hóa đơn'),
        content: TextField(controller: ctrl, autofocus: true, decoration: const InputDecoration(hintText: 'Nhập ghi chú cho toàn bộ hóa đơn...')),
        actions: [
          ElevatedButton(
            onPressed: () {{
              setState(() =""" + f"""> activeOrderNote = ctrl.text.trim());
              Navigator.pop(context);
            }},
            child: const Text('Lưu'),
          ),
        ],
      ),
    );
  }}

  void _showMergeOrderModal() {{
    showDialog(
      context: context,
      builder: (context) =""" + f"""> AlertDialog(
        title: const Text('Gộp với hóa đơn đang mở khác'),
        content: SizedBox(
          width: 420,
          height: 300,
          child: ListView(
            children: openOrders
                .where((o) =""" + f"""> o['id'] != activeOrderId)
                .map((o) =""" + f"""> ListTile(
                      title: Text('${{o['table_name']}} - ${{fmt.format(o['total'])}}₫'),
                      trailing: ElevatedButton(
                        onPressed: () async {{
                          final otherItems = jsonDecode(o['items_json'].toString());
                          for (var it in otherItems) cart.add(it);
                          final db = await LocalDb.db;
                          await db.delete('orders', where: 'id = ?', whereArgs: [o['id']]);
                          await _loadData();
                          setState(() {{}});
                          if (mounted) Navigator.pop(context);
                        }},
                        child: const Text('Gộp vào đây'),
                      ),
                    ))
                .toList(),
          ),
        ),
      ),
    );
  }}
}}
"""

with open("lib/main.dart", "w", encoding="utf-8") as f:
    f.write(dart_code)

print("⏳ 2/2: Đang biên dịch ra file LaChuoi_Ocha_v11.apk...")
res = subprocess.run(["flutter", "build", "apk", "--release", "--no-tree-shake-icons"], capture_output=True, text=True)

apk_path = "/content/pos_sunmi_t2/build/app/outputs/flutter-apk/app-release.apk"
if os.path.exists(apk_path):
    final_apk = "/content/LaChuoi_Ocha_v11.apk"
    os.system(f"cp {apk_path} {final_apk}")
    print("✅ HOÀN TẤT! Đang tự động tải file LaChuoi_Ocha_v11.apk về máy của bạn...")
    from google.colab import files
    files.download(final_apk)
else:
    print("❌ CHI TIẾT LỖI:")
    print(res.stdout[-2000:])
    print(res.stderr[-2000:])

⏳ 1/2: Đang chuẩn bị môi trường biên dịch cho bản v11.0...
⏳ 2/2: Đang biên dịch ra file LaChuoi_Ocha_v11.apk...
✅ HOÀN TẤT! Đang tự động tải file LaChuoi_Ocha_v11.apk về máy của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>